# `02_main_analysis.ipynb`

## 日本語
このNotebookは、論文再現のための3段階ワークフローの第2部です。  
`01_data_preprocessing.ipynb` で作成した部門統合済みの地域間産業連関表を用いて、論文の主要な実証分析を実行します。  
`../output/tables/` から統合済み表を読み込み、Skyline分析、構造X-ray分析、漏出分解、長期比較、関連する集計表の作成など、論文の中核的な定量結果を再現します。  
出力結果は `../output/` 以下に、集計表・中間結果・図表作成用データとして保存されます。

## English
This notebook is the second part of the three-step reproduction workflow for the paper.  
It performs the main empirical analysis using the sector-integrated interregional input–output tables produced in `01_data_preprocessing.ipynb`.  
The notebook reads the integrated tables from `../output/tables/` and reproduces the core quantitative results of the paper, including Skyline analysis, structural X-ray analysis, leakage decomposition, long-run comparisons, and related summary tables.  
Outputs are saved under `../output/`, including tables, intermediate results, and analysis-ready data for figures.

## 初期設定

In [ ]:
# =========================================================
# 1. 環境構築とライブラリのインポート
# =========================================================
!pip install -q japanize-matplotlib adjustText matplotlib-fontja

import os
import shutil
import warnings
import glob
from PIL import Image

import numpy as np
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import japanize_matplotlib
from adjustText import adjust_text

import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import ConnectionPatch, Patch, Rectangle, FancyArrowPatch
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import imageio.v2 as imageio

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# フォント・スタイルの基本設定
plt.rcParams.update({
    'font.size': 14, 'axes.titlesize': 22, 'axes.labelsize': 18,
    'xtick.labelsize': 16, 'ytick.labelsize': 16, 'legend.fontsize': 16,
    'figure.titlesize': 24, 'axes.unicode_minus': False,
    'font.family': 'sans-serif'
})
plt.rcParams['font.sans-serif'] = ['Hiragino Maru Gothic Pro', 'Yu Gothic', 'Meiryo', 'IPAexGothic', 'sans-serif']

# =========================================================
# 2. 共通ディレクトリパス設定
# =========================================================

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
BASE_DIR = ROOT_DIR
DATA_DIR = os.path.join(ROOT_DIR, 'output', 'tables')
SAVE_DIR = os.path.join(ROOT_DIR, 'output')
SHAPEFILE_RAW = os.path.join(ROOT_DIR, 'data', 'raw', 'municipality_shapefile', 'region.shp')

# 必要なサブディレクトリの一括作成
sub_dirs = ['skyline', 'leakage', 'xray', 'results', 'gif', 'frames', 'tables', 'map', 'longterm', 'processed_map', 'bridge_analysis', 'interregional_comparison']
for d in sub_dirs:
    os.makedirs(os.path.join(SAVE_DIR, d), exist_ok=True)

# 年次とファイル名の対応（※renameなしのファイル名に変更）
YEAR_FILES = {
    '1985': os.path.join(DATA_DIR, 'S60_integrated.xlsx'),
    '1990': os.path.join(DATA_DIR, 'H2_integrated.xlsx'),
    '1995': os.path.join(DATA_DIR, 'H7_integrated.xlsx'),
    '2005': os.path.join(DATA_DIR, 'H17_integrated.xlsx')
}
WAREKI_MAP = {'1985': '1985/S60', '1990': '1990/H2', '1995': '1995/H7', '2005': '2005/H17'}

# =========================================================
# 3. 共通マスタデータ・定数定義
# =========================================================
ALL_REGIONS = ['hokkaido', 'tohoku', 'kanto', 'chubu', 'kinki', 'chugoku', 'shikoku', 'kyushu', 'okinawa']
REGIONS_JA = ['北海道', '東北', '関東', '中部', '近畿', '中国', '四国', '九州', '沖縄']
REGIONS_EN = ['Hokkaido', 'Tohoku', 'Kanto', 'Chubu', 'Kinki', 'Chugoku', 'Shikoku', 'Kyushu', 'Okinawa']

# SECTOR_EN = {
#     '農林水産業': 'Agriculture, forestry and fishery', '鉱業': 'Mining', '飲食料品': 'Beverages and foods',
#     '繊維工業製品': 'Textile products', '製材・木製品・家具': 'Lumber and wood products',
#     'パルプ・紙・板紙・加工紙': 'Pulp, paper and paper products', '出版・印刷': 'Printing, plate making and book binding',
#     '化学製品': 'Chemical products', '石油・石炭製品': 'Petroleum and coal products',
#     'プラスチック製品': 'Plastic products', 'その他の製造工業製品': 'Miscellaneous manufacturing products',
#     '窯業・土石製品': 'Ceramic, stone and clay products', '鉄鋼': 'Iron and steel',
#     '非鉄金属': 'Non-ferrous metals', '金属製品': 'Metal products', '一般機械': 'General-purpose machinery',
#     '電気機械': 'Electrical machinery', '自動車': 'Motor vehicles', 'その他輸送用機械': 'Other transport equipment',
#     '精密機械': 'Precision instruments', '建設': 'Construction', '電力': 'Electricity',
#     'ガス・熱供給': 'Gas and heat supply', '水道・廃棄物処理': 'Water supply and waste management service',
#     '商業': 'Commerce', '金融・保険': 'Finance and insurance', '不動産': 'Real estate',
#     '運輸': 'Transport and postal services', '情報通信': 'Information and communications',
#     '公務': 'Public administration', '教育・研究': 'Education and research',
#     '医療・保健・社会保障・介護': 'Medical, health care and welfare', '広告・対事業所サービス': 'Business services',
#     '対個人サービス': 'Personal services', 'その他': 'Activities not elsewhere classified'
# }

SECTOR_EN = {
    '農林水産業': 'Agri, Forest, Fishery', '鉱業': 'Mining', '飲食料品': 'Beverages/Foods',
    '繊維工業製品': 'Textiles', '製材・木製品・家具': 'Lumber/Wood',
    'パルプ・紙・板紙・加工紙': 'Pulp/Paper', '出版・印刷': 'Printing',
    '化学製品': 'Chemicals', '石油・石炭製品': 'Petroleum/Coal',
    'プラスチック製品': 'Plastics', 'その他の製造工業製品': 'Misc. Mfg',
    '窯業・土石製品': 'Ceramics/Stone', '鉄鋼': 'Iron/Steel',
    '非鉄金属': 'Non-ferrous metals', '金属製品': 'Metal products', '一般機械': 'General machinery',
    '電気機械': 'Electrical machinery', '自動車': 'Motor vehicles', 'その他輸送用機械': 'Other transport equip.',
    '精密機械': 'Precision instruments', '建設': 'Construction', '電力': 'Electricity',
    'ガス・熱供給': 'Gas/Heat', '水道・廃棄物処理': 'Water/Waste',
    '商業': 'Commerce', '金融・保険': 'Finance/Insurance', '不動産': 'Real estate',
    '運輸': 'Transport', '情報通信': 'IT/Telecom',
    '公務': 'Public admin.', '教育・研究': 'Education/Research',
    '医療・保健・社会保障・介護': 'Medical/Welfare', '広告・対事業所サービス': 'Business services',
    '対個人サービス': 'Personal services', 'その他': 'Others'
}

MACRO_GROUPS = {
    '重化学工業': ['化学製品', '石油・石炭製品', 'プラスチック製品', '窯業・土石製品', '鉄鋼', '非鉄金属', '金属製品', '一般機械', '電気機械', '自動車', 'その他輸送用機械', '精密機械'],
    '軽工業': ['農林水産業', '鉱業', '飲食料品', '繊維工業製品', '製材・木製品・家具', 'パルプ・紙・板紙・加工紙', '出版・印刷', 'その他の製造工業製品'],
    '建設・インフラ': ['建設', '電力', 'ガス・熱供給', '水道・廃棄物処理'],
    'サービス・その他': ['商業', '金融・保険', '不動産', '運輸', '情報通信', '公務', '教育・研究', '医療・保健・社会保障・介護', '広告・対事業所サービス', '対個人サービス', 'その他']
}
MACRO_EN = {'重化学工業': 'Heavy Chemical Industry', '軽工業': 'Light Industry', '建設・インフラ': 'Construction & Infrastructure', 'サービス・その他': 'Services & Others', '全産業': 'All Industries'}

PALETTES = {
    'color': {
        'sky_prod': '#87CEEB', 'sky_exp': '#D3D3D3', 'sky_imp_edge': '#FF0000', 'sky_line': '#FF0000',
        'leakage': ['#66c2a5', '#fc8d62', '#8da0cb', '#a6d854', '#e78ac3'],
        'xray_exp': '#E0E0E0', 'xray_leaks': ['#fc8d62', '#8da0cb', '#a6d854', '#e78ac3'], 'xray_line': '#FF0000',
        'hatches': ['', '', '', '', '']
    },
    'bw': {
        'sky_prod': 'white', 'sky_exp': '#E0E0E0', 'sky_imp_edge': 'black', 'sky_line': 'black',
        'leakage': ['white', '#EEEEEE', '#CCCCCC', '#AAAAAA', '#777777'],
        'xray_exp': 'white', 'xray_leaks': ['#EEEEEE', '#CCCCCC', '#AAAAAA', '#777777'], 'xray_line': 'black',
        'hatches': ['', '//', '..', 'oo', 'xx']
    }
}

BW_STYLES = {
    'hokkaido': {'marker': 'o', 'ls': '-',  'color': '#888888'}, 'tohoku':   {'marker': 's', 'ls': '--', 'color': '#888888'},
    'kanto':    {'marker': '^', 'ls': ':',  'color': '#444444'}, 'chubu':    {'marker': 'D', 'ls': '-.', 'color': '#888888'},
    'kinki':    {'marker': 'v', 'ls': '-',  'color': '#888888'}, 'chugoku':  {'marker': '<', 'ls': '--', 'color': '#888888'},
    'shikoku':  {'marker': 'P', 'ls': '-',  'color': '#000000'}, 'kyushu':   {'marker': 'X', 'ls': '-.', 'color': '#888888'},
    'okinawa':  {'marker': '*', 'ls': ':',  'color': '#888888'}
}

TRANSLATIONS = {
    'ja': {
        'years': "年",
        'regions': dict(zip(ALL_REGIONS, REGIONS_JA)),
        # --- 支店経済化分析用ラベル ---
        'dep_title': "地域別 関東のプロデューサー・サービスへの依存度推移",
        'dep_ylabel': "関東への機能的依存度（誘発係数×100）",
        'cv_title': "産業構造の特化係数 (Skyline CV) の推移",
        'cv_ylabel': "Skyline CV (特化度)",
        'cv_label': "Skyline CV (特化度)",
        'ent_title': "産業構造の多様性 (Entropy) の推移",
        'ent_ylabel': "Entropy (多様性)",
        'entropy_label': "Entropy (多様性)",
        'idx_title': "四国：経済構造の特化と多様性の共進化",
        'heatmap_title': "産業部門別 関東への依存度ヒートマップ",
        'heatmap_xlabel': "地域",
        'heatmap_ylabel': "産業部門",
        'dep_shikoku_title': "四国：主要サービス部門の関東依存度推移",
        'avg_label': "全産業平均",
        'dist_title': "地域別 関東依存度の分布 (Boxplot)",
        'ind_map': {'情報通信': '情報通信', '金融・保険': '金融・保険', '広告・対事業所サービス': '対事業所サービス', '卸売・小売': '商業'}
    },
    'en': {
        'years': "Year",
        'regions': {r: r.capitalize() for r in ALL_REGIONS},
        # --- Branch Economy Labels ---
        'dep_title': "Regional Economic Dependency on Kanto (Weighted Avg)",
        'dep_ylabel': "Dependency on Kanto Producer Services \n (coefficient ×100)",
        'cv_title': "Evolution of Industrial Specialization (Skyline CV)",
        'cv_ylabel': "Skyline CV (Specialization)",
        'cv_label': "Skyline CV",
        'ent_title': "Evolution of Industrial Diversity (Entropy)",
        'ent_ylabel': "Entropy (Diversity)",
        'entropy_label': "Entropy",
        'idx_title': "Shikoku: Co-evolution of Specialization & Diversity",
        'heatmap_title': "Heatmap of Dependency on Kanto by Sector",
        'heatmap_xlabel': "Region",
        'heatmap_ylabel': "Industrial Sector",
        'dep_shikoku_title': "Shikoku: Dependency of Key Service Sectors",
        'avg_label': "Regional Average(Weighted)",
        'dist_title': "Distribution of Dependency on Kanto (Boxplot)",
        'ind_map': {'情報通信': 'Info & Comm', '金融・保険': 'Finance & Insurance', '広告・対事業所サービス': 'Business Services', '卸売・小売': 'Commerce'}
    }
}

FINAL_FIGURES = [
    "shikoku_bridges_color.png",
    "region_map_paper_style.png",
    "shikoku_macro_ss_en_color.png",
    "compare_cv_en_color.png",
    "compare_entropy_en_color.png",
    "skyline_shikoku_1985_en_color.png",
    "skyline_shikoku_1990_en_color.png",
    "skyline_shikoku_1995_en_color.png",
    "skyline_shikoku_2005_en_color.png",
    "shikoku_balance_transition_en_color.png",
    "shikoku_total_circulation_structure_all_en_color.png",
    "shikoku_leakage_ratio_only_all_en_color.png",
    "compare_dep_avg_en_color.png",
    "shikoku_macro_dep_with_avg_en_color.png",
    "shikoku_temporal_decomposition_en_color.png",
    "regional_comparison_combined_en_color.png",
    "structural_transformation_scatter_1985_2005_en_color.png",
    "compare_total_ssr_en_color.png",
]

GROUP_FILEKEY = {
    '全産業': 'all',
    '製造業': 'manufacturing',
    '重化学工業': 'heavy_chemical',
    '軽工業': 'light_industry',
    '建設・インフラ': 'construction_infra',
    'サービス・その他': 'services_other',
}

# =========================================================
# 0-X. 実行設定（全体制御）
# =========================================================

RUN_CONFIG = {
    'styles': {
        'skyline_xray': ['color', 'bw'],
        'longterm': ['color'],
        'branch_economy': ['color', 'bw'],
        'bridge': ['color'],
        'sharpening_summary': ['color'],
    },
    'languages': {
        'skyline_xray': ['ja', 'en'],
        'longterm': ['en'],
        'branch_economy': ['en'],
        'bridge': ['en'],
        'sharpening_summary': ['en'],
    }
}


In [ ]:
def initialize_full_target_config():
    conf = {}
    for i, key in enumerate(ALL_REGIONS):
        if i == 2: g1, g2 = [4], [3]
        elif i == 4: g1, g2 = [2], [3]
        else: g1, g2 = [2], [4]
        g1_ja, g2_ja = REGIONS_JA[g1[0]], REGIONS_JA[g2[0]]
        g1_en, g2_en = REGIONS_EN[g1[0]], REGIONS_EN[g2[0]]
        conf[key] = {
            'idx': i, 'name_ja': REGIONS_JA[i], 'name_en': REGIONS_EN[i],
            'leakage_groups': {
                'ja': ['域内', f'{g1_ja}向け', f'{g2_ja}向け', '国内他地域向け', '海外向け'],
                'en': ['Intraregional', f'To {g1_en}', f'To {g2_en}', 'To Rest of Japan', 'To Overseas']
            },
            'group_indices': {'g1': g1, 'g2': g2, 'others': [j for j in range(9) if j != i and j not in g1 and j not in g2]}
        }
    return conf

TARGET_CONFIG = initialize_full_target_config()

def load_and_extract(filepath, num_regions=9, num_sectors=35):
    # 読み込み（ヘッダーなしで全データを読み込む）
    df_raw = pd.read_excel(filepath, header=None)

    # --- 前処理 ---
    # 【行の削除】
    # 0行目: タイトル (昭和60...)
    # 1行目: サブタイトル (45部門...)
    # 2行目: メタヘッダー (地域(行側)...)
    # 4行目: 列コード (010, 040...)
    df_raw = df_raw.drop([0, 1, 2, 4], axis=0).reset_index(drop=True)

    # 【列の削除】
    # 0列目: 地域コード (1, 1...)
    # 2列目: 行コード (010, 040...)
    df_raw = df_raw.drop(df_raw.columns[[0, 2]], axis=1)

    # インデックスとカラム名を整える
    df_raw.columns = range(df_raw.shape[1])
    # ------------------------------------

    row_block, col_block = 45, 48
    s_rows = [2 + r * row_block + s for r in range(num_regions) for s in range(num_sectors)]
    s_cols = [2 + r * col_block + s for r in range(num_regions) for s in range(num_sectors)]

    def to_num(df): return df.replace(['-', ' ', ''], 0).apply(pd.to_numeric).fillna(0).values

    # Z項の読み出し
    Z = to_num(df_raw.iloc[s_rows, s_cols])

    # X項（総生産）の読み出し
    X = to_num(df_raw.iloc[s_rows, 482])

    F_list, M = [], np.zeros(num_regions * num_sectors)
    for r in range(num_regions):
        F_list.append(to_num(df_raw.iloc[s_rows, slice(2 + r * col_block + 36, 2 + r * col_block + 41)]).sum(axis=1))
        r_slice = slice(r * num_sectors, (r + 1) * num_sectors)
        M[r_slice] = np.abs(to_num(df_raw.iloc[s_rows[r_slice], 2 + r * col_block + 46]))

    return Z, X, F_list, M, df_raw.iloc[2:2+num_sectors, 1].values

def analyze_structure(Z, X, F_list, M, sector_names, target_key):
    cfg = TARGET_CONFIG[target_key]
    t_idx, n, n_sec = cfg['idx'], Z.shape[0], 35
    sh_slice = slice(t_idx * n_sec, (t_idx + 1) * n_sec)

    X_safe = np.where(X == 0, 1e-9, X)
    A = Z @ np.diag(1 / X_safe)
    Total_D = (A @ X) + np.sum(F_list, axis=0)
    m = np.clip(M / np.where(Total_D <= 0, 1, Total_D), 0, 1)
    L_dom = np.linalg.inv(np.eye(n) - (np.diag(1 - m) @ A))
    X_ind = L_dom @ F_list[t_idx]
    D_ri = np.array([X_ind[i::n_sec].sum() for i in range(n_sec)])

    def sum_leak(indices): return sum(L_dom[i*n_sec:(i+1)*n_sec, sh_slice].sum(axis=0) for i in indices)

    return {
        'skyline': pd.DataFrame({'sector': sector_names, 'share': D_ri/np.sum(D_ri) if np.sum(D_ri) else 0, 'xd': D_ri, 'x': X[sh_slice], 'ss': X[sh_slice]/np.where(D_ri==0, 1, D_ri)}),
        'leakage': pd.DataFrame({'sector': sector_names, 'intra': L_dom[sh_slice, sh_slice].sum(axis=0), 'g1': sum_leak(cfg['group_indices']['g1']),
                                 'g2': sum_leak(cfg['group_indices']['g2']), 'others': sum_leak(cfg['group_indices']['others']),
                                 'overseas': m[sh_slice] + (m @ (A @ L_dom[:, sh_slice]))})
    }

def verify_preprocessing_exact(raw_path):
    print(f"\n=== Verification for {os.path.basename(raw_path)} ===")
    if not os.path.exists(raw_path):
        print("File not found.")
        return

    df_raw = pd.read_excel(raw_path, header=None)
    print(f"Original Shape: {df_raw.shape}")

    # 提案した削除ロジック
    df_proc = df_raw.drop([0, 1, 2, 4], axis=0).reset_index(drop=True)
    df_proc = df_proc.drop(df_proc.columns[[0, 2]], axis=1)
    df_proc.columns = range(df_proc.shape[1])

    print(f"Processed Shape: {df_proc.shape}")

    # ターゲット（rename版）の形状 (452, 483) と一致するか
    target_shape = (452, 483)
    if df_proc.shape == target_shape:
        print("✅ Shapes Match! (452, 483)")

        # データ開始位置の値をチェック（数値であることを確認）
        # 行2, 列2 が最初のデータセル（北海道・農林水産業の自己消費など）
        first_val = df_proc.iloc[2, 2]
        print(f"Value at Data Start [2, 2]: {first_val}")
        if isinstance(first_val, (int, float, np.number)):
            print("✅ Data start looks numeric.")
        else:
            print(f"⚠️ Warning: Data start is not numeric (Type: {type(first_val)})")
    else:
        print(f"⚠️ Shape Mismatch. Expected {target_shape}, got {df_proc.shape}")
        print("削除ロジックの再調整が必要です。")

# 実行
raw_file_1985 = YEAR_FILES['1985']
verify_preprocessing_exact(raw_file_1985)

In [ ]:
DATA_CACHE = {}
STRUCTURE_CACHE = {}

def get_year_data(year):
    year = str(year)
    if year not in DATA_CACHE:
        DATA_CACHE[year] = load_and_extract(YEAR_FILES[year])
    return DATA_CACHE[year]

def get_structure(year, target_key):
    year = str(year)
    cache_key = (year, target_key)
    if cache_key not in STRUCTURE_CACHE:
        Z, X, F_list, M, names = get_year_data(year)
        STRUCTURE_CACHE[cache_key] = analyze_structure(Z, X, F_list, M, names, target_key)
    return STRUCTURE_CACHE[cache_key]


Z, X, F_list, M, names = get_year_data('1985')
print(names[:5])  # ['農林水産業', '鉱業', '飲食料品', ...] と出ればOK

In [ ]:
def calc_weighted_ssr(sky_df):
    return float((sky_df['share'] * sky_df['ss']).sum())

def calc_cv_ssr(sky_df):
    ss = sky_df['ss'].values
    mean_ss = np.mean(ss)
    return float(np.std(ss) / mean_ss) if mean_ss != 0 else np.nan

def calc_export_base_share(sky_df, threshold=1.0):
    return float(sky_df.loc[sky_df['ss'] > threshold, 'share'].sum())

def calc_sharpening_index(df_t0, df_t1):
    comp = pd.merge(
        df_t0[['sector', 'share', 'ss']],
        df_t1[['sector', 'share', 'ss']],
        on='sector', suffixes=('_t0', '_t1')
    )
    comp['d_share'] = comp['share_t1'] - comp['share_t0']
    comp['d_ss'] = comp['ss_t1'] - comp['ss_t0']

    def _phi(row):
        if row['share_t0'] == 0:
            return np.nan
        denom = abs(row['d_share'] / row['share_t0'])
        if denom == 0:
            return np.nan
        return (row['d_ss'] / row['ss_t0']) / denom if row['ss_t0'] != 0 else np.nan

    comp['phi'] = comp.apply(_phi, axis=1)
    return comp[['sector', 'share_t0', 'share_t1', 'ss_t0', 'ss_t1', 'd_share', 'd_ss', 'phi']]

In [ ]:
# =========================================================
# デフレーター計算および管理用クラス (Excel直接読み込み版)
# =========================================================
def map_185_to_35_ja(code):
    """総務省185部門コードを本分析の35部門名に変換"""
    try: c = int(code)
    except: return None
    if 111 <= c <= 312: return '農林水産業'
    if 611 <= c <= 721: return '鉱業'
    if 1111 <= c <= 1141: return '飲食料品'
    if 1511 <= c <= 1529: return '繊維工業製品'
    if 1611 <= c <= 1711: return '製材・木製品・家具'
    if 1811 <= c <= 1829: return 'パルプ・紙・板紙・加工紙'
    if 1911 <= c <= 1911: return '出版・印刷'
    if 2011 <= c <= 2079: return '化学製品'
    if 2111 <= c <= 2121: return '石油・石炭製品'
    if 2211 <= c <= 2211: return 'プラスチック製品'
    if 2511 <= c <= 2599: return '窯業・土石製品'
    if 2611 <= c <= 2649: return '鉄鋼'
    if 2711 <= c <= 2722: return '非鉄金属'
    if 2811 <= c <= 2899: return '金属製品'
    if 3011 <= c <= 3112: return '一般機械'
    if 3211 <= c <= 3421: return '電気機械'
    if 3511 <= c <= 3541: return '自動車'
    if 3611 <= c <= 3629: return 'その他輸送用機械'
    if 3711 <= c <= 3719: return '精密機械'
    if (2311 <= c <= 2412) or (3911 <= c <= 3919): return 'その他の製造工業製品'
    if 4111 <= c <= 4132: return '建設'
    if 5111 <= c <= 5111: return '電力'
    if 5121 <= c <= 5122: return 'ガス・熱供給'
    if 5211 <= c <= 5212: return '水道・廃棄物処理'
    if 6111 <= c <= 6112: return '商業'
    if 6211 <= c <= 6212: return '金融・保険'
    if 6411 <= c <= 6421: return '不動産'
    if 7111 <= c <= 7189: return '運輸'
    if 7311 <= c <= 7351: return '情報通信'
    if 8111 <= c <= 8112: return '公務'
    if 8211 <= c <= 8222: return '教育・研究'
    if 8311 <= c <= 8314: return '医療・保健・社会保障・介護'
    if 8511 <= c <= 8519: return '広告・対事業所サービス'
    if 8611 <= c <= 8619: return '対個人サービス'
    if c in [8411, 8900, 9000]: return 'その他'
    return None

class DeflatorManager:
    def __init__(self):
        self.deflators = None

    def calculate_deflators(self, s60h2h7_path, h7h12h17_path):
        print(">>> Excelファイルからデフレーターを算出しています...")

        def extract_s60(filepath, sheet_name, val_col):
            df = pd.read_excel(filepath, sheet_name=sheet_name, header=None)
            col_codes = df.iloc[1, 2:].values
            row_idx = df[df[0].astype(str).str.contains('9700', na=False)].index[0]
            values = df.iloc[row_idx, 2:].values
            res = pd.DataFrame({'code': col_codes, val_col: values})
            res = res.dropna(subset=['code'])
            res['code'] = res['code'].astype(str).str.strip().str.zfill(4)
            res['sector'] = res['code'].apply(map_185_to_35_ja)
            res[val_col] = pd.to_numeric(res[val_col], errors='coerce').fillna(0)
            return res.groupby('sector')[val_col].sum()

        # 各シートからデータを抽出
        s60_n = extract_s60(s60h2h7_path, '昭和60年名目', 'S60_nom')
        s60_r = extract_s60(s60h2h7_path, '昭和60年実質', 'S60_real')
        h2_n = extract_s60(s60h2h7_path, '平成2年名目', 'H2_nom')
        h2_r = extract_s60(s60h2h7_path, '平成2年実質', 'H2_real')

        df_h7 = pd.read_excel(h7h12h17_path, sheet_name='統合小分類', header=None)
        prod_df = df_h7[df_h7[3] == "国内生産額"].copy()
        prod_df['code'] = prod_df[0].astype(str).str.strip().str.zfill(4)
        prod_df['sector'] = prod_df['code'].apply(map_185_to_35_ja)
        for c, col_name in zip([5, 8, 17], ['H7_nom', 'H7_real', 'H17_nom']):
            prod_df[col_name] = pd.to_numeric(prod_df[c], errors='coerce').fillna(0)
        agg_h7 = prod_df.groupby('sector')[['H7_nom', 'H7_real', 'H17_nom']].sum()

        final_df = pd.concat([s60_n, s60_r, h2_n, h2_r, agg_h7], axis=1).fillna(0)

        # デフレーターの計算とチェーニング (2005年基準)
        def_95_to_05 = final_df['H7_nom'] / final_df['H7_real'].replace(0, np.nan)
        def_85_to_95 = final_df['S60_nom'] / final_df['S60_real'].replace(0, np.nan)
        def_90_to_95 = final_df['H2_nom'] / final_df['H2_real'].replace(0, np.nan)

        self.deflators = pd.DataFrame({
            '1985': def_85_to_95 * def_95_to_05,
            '1990': def_90_to_95 * def_95_to_05,
            '1995': def_95_to_05,
            '2005': 1.0
        }).fillna(1.0)
        print(">>> デフレーターの算出が完了しました。")

# グローバルインスタンスの生成 (ファイルパスは適宜調整してください)
GLOBAL_DEFLATOR_MANAGER = DeflatorManager()
GLOBAL_DEFLATOR_MANAGER.calculate_deflators(
    s60h2h7_path=os.path.join(DATA_DIR, "S60H2H7.xls"),
    h7h12h17_path=os.path.join(DATA_DIR, "H7H12H17.xls")
)
# 1. 論文の部門番号順（1〜35）を定義
sector_order = [
    '農林水産業', '鉱業', '飲食料品', '繊維工業製品', '製材・木製品・家具',
    'パルプ・紙・板紙・加工紙', '出版・印刷', '化学製品', '石油・石炭製品', 'プラスチック製品',
    '窯業・土石製品', '鉄鋼', '非鉄金属', '金属製品', '一般機械',
    '電気機械', '自動車', 'その他輸送用機械', '精密機械', 'その他の製造工業製品',
    '建設', '電力', 'ガス・熱供給', '水道・廃棄物処理', '商業',
    '金融・保険', '不動産', '運輸', '情報通信', '公務',
    '教育・研究', '医療・保健・社会保障・介護', '広告・対事業所サービス', '対個人サービス', 'その他'
]

# 2. 定義した順に並び替え（マスタに存在しない部門は除外）
available_order = [s for s in sector_order if s in GLOBAL_DEFLATOR_MANAGER.deflators.index]
sorted_deflators = GLOBAL_DEFLATOR_MANAGER.deflators.reindex(available_order)

# 3. 部門番号を付与して表示
sorted_deflators_display = sorted_deflators.copy()
sorted_deflators_display.index = [f"{i+1:02d}_{name}" for i, name in enumerate(sorted_deflators_display.index)]

print("\n--- 部門番号順 デフレーター一覧 (2005年=1.0) ---")
display(sorted_deflators_display)


# CSVとして保存したい場合
# GLOBAL_DEFLATOR_MANAGER.deflators.to_csv(os.path.join(SAVE_DIR, "calculated_deflators.csv"))

def calculate_real_factor_decomposition(
        X_nom_t0, X_nom_t1,
        XD_nom_t0, XD_nom_t1,
        deflators_t0, deflators_t1):
    """
    実質シェアに基づく要因分解（Appendix用頑健性チェック）
    SSRは名目・実質で数学的に一致するため名目XDから算出する。
    実質化するのはwiの計算に使うXDのみ。
    """
    # pd.Series対応
    def to_arr(v): return v.values if hasattr(v, 'values') else np.asarray(v)
    p0, p1 = to_arr(deflators_t0), to_arr(deflators_t1)
    xd0, xd1 = to_arr(XD_nom_t0), to_arr(XD_nom_t1)
    x0,  x1  = to_arr(X_nom_t0),  to_arr(X_nom_t1)

    # SSRは名目値から算出（実質化不要・価格不変）
    ssr_t0 = x0 / np.where(xd0 == 0, 1, xd0)
    ssr_t1 = x1 / np.where(xd1 == 0, 1, xd1)

    # 需要誘発額のみデフレート → 実質シェア
    xd_real_t0 = xd0 / np.where(p0 == 0, 1, p0)
    xd_real_t1 = xd1 / np.where(p1 == 0, 1, p1)

    w_real_t0 = xd_real_t0 / xd_real_t0.sum()
    w_real_t1 = xd_real_t1 / xd_real_t1.sum()

    mean_ssr    = (ssr_t0 + ssr_t1) / 2
    mean_w_real = (w_real_t0 + w_real_t1) / 2

    real_share_effect = mean_ssr    * (w_real_t1 - w_real_t0)
    real_rate_effect  = mean_w_real * (ssr_t1 - ssr_t0)

    return real_share_effect, real_rate_effect


In [ ]:
# =========================================================
# Appendix用 デフレーターのLaTeXテーブル出力コード
# =========================================================
print(r"\begin{table}[htbp]")
print(r"    \centering")
print(r"    \caption{National Sectoral Output Deflators (1985--2005, Base Year 2005 = 1.0)}")
print(r"    \label{tab:deflators}")
print(r"    \vspace{0.2cm}")
print(r"    \scriptsize")
print(r"    \begin{tabular}{llcccc}")
print(r"        \toprule")
print(r"        \textbf{No.} & \textbf{Sector} & \textbf{1985} & \textbf{1990} & \textbf{1995} & \textbf{2005} \\")
print(r"        \midrule")

for idx_str, row in sorted_deflators_display.iterrows():
    # '01_農林水産業' から番号と名前に分割して英語名を取得
    num = int(idx_str.split('_')[0])
    ja_name = idx_str.split('_')[1]
    en_name = SECTOR_EN.get(ja_name, ja_name)

    # データをフォーマット
    v85 = f"{row['1985']:.3f}"
    v90 = f"{row['1990']:.3f}"
    v95 = f"{row['1995']:.3f}"
    v05 = f"{row['2005']:.3f}"

    print(f"        {num} & {en_name} & {v85} & {v90} & {v95} & {v05} \\\\")

print(r"        \bottomrule")
print(r"    \end{tabular}")
print(r"    \vspace{0.1cm}")
print(r"    \begin{flushleft}")
print(r"    \scriptsize * Note: Derived from the linked interregional input-output tables (Ministry of Internal Affairs and Communications). Deflators are calculated as the ratio of nominal to real domestic output.")
print(r"    \end{flushleft}")
print(r"\end{table}")

In [ ]:
import difflib

def verify_sector_names_strict():
    print("=== 産業部門名マッチング検証 ===")

    # 1. 1985年のデータをサンプルとして読み込む
    year = '2005'
    filepath = YEAR_FILES[year]

    if not os.path.exists(filepath):
        print(f"エラー: ファイルが見つかりません {filepath}")
        return

    # 独自に読み込み（前処理ロジックを再現）
    df_raw = pd.read_excel(filepath, header=None)
    df_raw = df_raw.drop([0, 1, 2, 4], axis=0).reset_index(drop=True)
    df_raw = df_raw.drop(df_raw.columns[[0, 2]], axis=1)

    # 実際のデータ上の名前（35部門分）を取得
    # ※あえて strip() せずに生の状態で取得して確認する
    raw_names = df_raw.iloc[2:2+35, 1].values

    # 文字列型に変換し、None等は空文字にする
    actual_names = [str(x) if pd.notnull(x) else "" for x in raw_names]

    # 2. エラーの原因となっている「サービス・その他」グループと比較
    target_group = MACRO_GROUPS['サービス・その他']

    print(f"\n【検証対象グループ】: サービス・その他 ({len(target_group)} 部門)")
    print("-" * 60)
    print(f"{'期待する名前':<20} | {'判定':<6} | {'詳細 / 近似候補'}")
    print("-" * 60)

    mismatch_count = 0

    for expected in target_group:
        # 完全一致があるかチェック
        if expected in actual_names:
            status = "OK"
            detail = ""
        else:
            status = "MISSING"
            mismatch_count += 1

            # 似ている名前を探す（空白ズレや表記ゆれを検知）
            matches = difflib.get_close_matches(expected, actual_names, n=1, cutoff=0.5)
            if matches:
                found = matches[0]
                # repr()を使って空白を可視化して表示
                detail = f"候補あり: {repr(found)} (期待値: {repr(expected)})"
            else:
                detail = "候補なし"

        print(f"{expected:<20} | {status:<6} | {detail}")

    print("-" * 60)

    # 3. 実際の全リストを表示（デバッグ用）
    print("\n【ファイル内の全産業名リスト (生データ)】")
    for i, name in enumerate(actual_names):
        print(f"{i:02d}: {repr(name)}")

    if mismatch_count > 0:
        print(f"\n⚠️ {mismatch_count} 件の不一致が見つかりました。")
        print("解決策: load_and_extract関数内で .str.strip() を使うか、MACRO_GROUPSの定義をファイル側に合わせてください。")
    else:
        print("\n✅ すべて一致しています。別の箇所に原因がある可能性があります。")

# 検証を実行
verify_sector_names_strict()

In [ ]:
def ensure_initialization():
    required_names = [
        'SAVE_DIR', 'YEAR_FILES', 'TARGET_CONFIG',
        'load_and_extract', 'analyze_structure',
        'GLOBAL_DEFLATOR_MANAGER'
    ]
    missing = [name for name in required_names if name not in globals()]
    if missing:
        raise RuntimeError(f"初期設定が未実行です。不足: {missing}")

## スカイライン・X-ray

In [ ]:
# =========================================================
# スカイライン分析（高さ固定・統合版）
# =========================================================

# --- 1. 個別設定 ---
HIGHLIGHT_SECTORS = ['パルプ・紙・板紙・加工紙', '非鉄金属', 'その他輸送用機械']

def get_plot_labels(df, lang):
    """言語設定に応じたラベル配列を返す"""
    if lang == 'en':
        return df['sector'].map(lambda x: SECTOR_EN.get(x, x)).values
    return df['sector'].values

def apply_highlight(ax, labels):
    """特定の産業ラベルを赤字・太字にする"""
    for tick_label in ax.get_xticklabels():
        txt = tick_label.get_text()
        # 日本語または英語名がハイライト対象に含まれるかチェック
        if txt in HIGHLIGHT_SECTORS or any(SECTOR_EN.get(h) == txt for h in HIGHLIGHT_SECTORS):
            tick_label.set_fontweight('bold')
            tick_label.set_color('red')

def plot_skyline_fixed(ax, sky_df, d_year, lang, style, ylim):
    p = PALETTES[style]
    l_map = {
        'ja': ['域内生産', '域外移輸出', '移輸入依存分', '自給率ライン', '自給率'],
        'en': ['Domestic Prod.', 'Regional Exports', 'Import Req.', 'SSR Line', 'Self-sufficiency Ratio']
    }

    # X軸の位置計算（累積シェア）
    df = sky_df.copy()
    widths = df['share'].values
    x_pos = np.concatenate([[0], np.cumsum(widths)[:-1]])
    ss_vals = df['ss'].values
    labels = get_plot_labels(df, lang)

    line_x, line_y = [], []
    for i, (w, ss) in enumerate(zip(widths, ss_vals)):
        cum_w = x_pos[i]
        sec_name = df.iloc[i]['sector']
        is_high = sec_name in HIGHLIGHT_SECTORS
        lw = 2.0 if is_high else 0.4

        # 生産（自給）部分
        ax.add_patch(Rectangle((cum_w, 0), w, min(ss, 1.0),
                               fc=p['sky_prod'], ec='black', lw=lw, zorder=2))

        # 移輸出部分
        if ss > 1.0:
            ax.add_patch(Rectangle((cum_w, 1.0), w, ss - 1.0,
                                   fc=p['sky_exp'], ec='black', lw=lw, zorder=2))

        # 移輸入部分
        if ss < 1.0:
            ax.add_patch(Rectangle((cum_w, ss), w, 1.0 - ss,
                                   fc='none', ec=p['sky_imp_edge'], lw=lw, hatch='\\\\\\\\\\\\\\\\', zorder=2))

        if is_high:
            ax.text(cum_w + w/2, ss + (ylim * 0.02), '★',
                    ha='center', color='red', fontsize=12, fontweight='bold')

        line_x.extend([cum_w, cum_w + w])
        line_y.extend([ss, ss])

    ax.plot(line_x, line_y, color=p['sky_line'], lw=2.5, label=l_map[lang][3], zorder=4)
    ax.axhline(1.0, color='black', ls='--', lw=1)

# --- 追加箇所: 20% (0.2) ごとの縦線（目安線）と数値ラベルを引く ---
    for v_line in [0.2, 0.4, 0.6, 0.8]:
        # 縦線を描画
        ax.axvline(x=v_line, color='gray', linestyle=':', linewidth=1.5, alpha=0.6, zorder=1)
        # 上部に数値を描画 (ylimの98%の高さに配置)
        ax.text(v_line, ylim * 0.98, f"{int(v_line * 100)}%",
                color='gray', fontsize=12, ha='center', va='top', alpha=0.8, zorder=1)
    # --------------------------------------------------

    ax.set_ylim(0, ylim)
    ax.set_xlim(0, 1)

    # X軸ラベル（中心位置に配置）
    tick_pos = x_pos + widths / 2
    ax.set_xticks(tick_pos)
    ax.set_xticklabels(labels, rotation=90, fontsize=13)
    apply_highlight(ax, labels)

    # タイトル・Y軸ラベル
    ax.tick_params(axis='y', labelsize=14)
    ax.set_ylabel(l_map[lang][4], fontsize=16)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.1f}'))

    # グラフ領域の左上 (x=0.02, y=0.96) に年次を表示
    ax.text(0.02, 0.96, d_year, transform=ax.transAxes,
            fontsize=32, fontweight='bold', color='gray', alpha=0.7,
            va='top', ha='left', zorder=10)

    # 凡例
    legend_elements = [
        Patch(facecolor=p['sky_prod'], edgecolor='black', label=l_map[lang][0]),
        Patch(facecolor=p['sky_exp'], edgecolor='black', label=l_map[lang][1]),
        Patch(facecolor='none', edgecolor=p['sky_imp_edge'], hatch='\\\\\\\\\\\\\\\\', label=l_map[lang][2]),
        Line2D([0], [0], color=p['sky_line'], lw=2.5, label=l_map[lang][3])
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=12)

def run_skyline_pipeline_fixed():
    print(">>> Starting Skyline Analysis (Fixed Height & Scale)...")

    # (中略: ディレクトリ作成等の初期設定部分は変更なし)
    skyline_dir = os.path.join(SAVE_DIR, 'skyline')
    frames_dir = os.path.join(SAVE_DIR, 'frames')
    target_keys = ['shikoku', 'kanto']
    years = sorted(YEAR_FILES.keys())

    print("  Step 1: Calculating Global Scales...")
    all_data = {k: {} for k in target_keys}
    y_max_scales = {k: 0 for k in target_keys}

    for year in years:
        if not os.path.exists(YEAR_FILES[year]): continue
        for t_key in target_keys:
            res = get_structure(year, t_key)
            all_data[t_key][year] = res['skyline']
            current_max = res['skyline']['ss'].max()
            if current_max > y_max_scales[t_key]:
                y_max_scales[t_key] = current_max

    print("  Step 2: Plotting and Saving...")
    for t_key in target_keys:
        # Y軸の上限設定（* 100 を削除）
        ylim = y_max_scales[t_key] * 1.15

        for year in years:
            if year not in all_data[t_key]:
                continue
            df_ss = all_data[t_key][year]

            for lang in RUN_CONFIG['languages']['skyline_xray']:
                for style in RUN_CONFIG['styles']['skyline_xray']:
                    fig, ax = plt.subplots(figsize=(20, 8))
                    d_year = WAREKI_MAP[year] if lang == 'ja' else year
                    plot_skyline_fixed(ax, df_ss, d_year, lang, style, ylim)
                    file_name = f"skyline_{t_key}_{year}_{lang}_{style}.png"
                    plt.savefig(os.path.join(skyline_dir, file_name), dpi=300, bbox_inches='tight')

                    frame_name = f"skyline_{t_key}_{year}_{lang}_{style}.png"
                    plt.savefig(os.path.join(frames_dir, frame_name), dpi=120, bbox_inches='tight')
                    plt.close()

    print(f"スカイライン分析が完了しました。出力先: {skyline_dir}")

run_skyline_pipeline_fixed()

In [ ]:
def plot_structural_xray(df_ss, df_leak, target_key, year, ylim, lang='ja', style='color'):
    cfg = TARGET_CONFIG[target_key]
    p = PALETTES[style]
    t = TRANSLATIONS[lang]

    df = pd.merge(df_ss, df_leak, on='sector')
    df['cum_share'] = df['share'].cumsum()
    df['prev_cum_share'] = df['cum_share'].shift(1).fillna(0)

    fig, ax = plt.subplots(figsize=(20, 8))

    stack_cols = ['intra', 'g1', 'g2', 'others', 'overseas']
    labels = cfg['leakage_groups'][lang]
    colors = p['xray_leaks'] if style == 'color' else p['leakage']
    hatches = p['hatches']

    for _, row in df.iterrows():
        width = row['share']
        left = row['prev_cum_share']
        bottom = 0

        total_ind = row[stack_cols].sum()
        if total_ind == 0: continue

        scale_factor = row['ss'] / total_ind

        for i, col in enumerate(stack_cols):
            val = row[col] * scale_factor
            if val <= 0: continue

            if col == 'intra':
                c = '#EEEEEE' if style == 'bw' else '#E0E0E0'
                h = ''
            else:
                c = colors[i-1] if (i-1) < len(colors) else 'gray'
                h = hatches[i] if style == 'bw' else ''

            ax.bar(left, val, width=width, bottom=bottom, align='edge',
                   color=c, edgecolor='black', linewidth=0.3, hatch=h)
            bottom += val

        if row['sector'] in HIGHLIGHT_SECTORS:
            ax.text(left + width/2, row['ss'] + (ylim * 0.02), '★',
                    ha='center', color='red', fontsize=12, fontweight='bold', zorder=5)

    ax.axhline(1.0, color='red' if style == 'color' else 'black', linestyle='--', linewidth=1.5)

    # --- 追加箇所: 20% (0.2) ごとの縦線（目安線）と数値ラベルを引く ---
    for v_line in [0.2, 0.4, 0.6, 0.8]:
        # 縦線を描画
        ax.axvline(x=v_line, color='gray', linestyle=':', linewidth=1.5, alpha=0.6, zorder=1)
        # 上部に数値を描画 (ylimの98%の高さに配置)
        ax.text(v_line, ylim * 0.98, f"{int(v_line * 100)}%",
                color='gray', fontsize=12, ha='center', va='top', alpha=0.8, zorder=1)
    # --------------------------------------------------

    ax.set_ylim(0, ylim)
    ax.set_xlim(0, 1.0)
    ax.xaxis.set_major_formatter(ticker.PercentFormatter(1.0))

    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.1f}'))
    ylabel_text = '自給率' if lang == 'ja' else 'Self-sufficiency Ratio'
    ax.set_ylabel(ylabel_text, fontsize=16)

    tick_pos = df['cum_share'] - (df['share'] / 2)
    ax.set_xticks(tick_pos)
    plot_labels = get_plot_labels(df, lang)
    ax.set_xticklabels(plot_labels, rotation=90, fontsize=13)
    apply_highlight(ax, plot_labels)

    r_name = t['regions'][target_key]
    d_year = WAREKI_MAP[year] if lang == 'ja' else year
    ax.text(0.02, 0.96, d_year, transform=ax.transAxes,
            fontsize=32, fontweight='bold', color='gray', alpha=0.7,
            va='top', ha='left', zorder=10)

    ax.legend(handles=[mpatches.Patch(facecolor='#E0E0E0', edgecolor='black', label=labels[0])] +
                      [mpatches.Patch(facecolor=colors[i-1], hatch=hatches[i] if style == 'bw' else '',
                       edgecolor='black', label=labels[i]) for i in range(1, len(labels))],
              loc='upper right', fontsize=12)

    save_dir = os.path.join(SAVE_DIR, 'xray')
    plt.savefig(os.path.join(save_dir, f"xray_{target_key}_{year}_{lang}_{style}.png"), dpi=300, bbox_inches='tight')
    plt.close()

def run_xray_pipeline():
    print(">>> Starting X-ray Analysis...")
    target_keys = ['shikoku', 'kanto']
    years = sorted(YEAR_FILES.keys())

    all_res = {k: {y: None for y in years} for k in target_keys}
    y_max_scales = {k: 0 for k in target_keys}

    for year in years:
        if not os.path.exists(YEAR_FILES[year]): continue
        for t_key in target_keys:
            res = get_structure(year, t_key)
            all_res[t_key][year] = res
            y_max_scales[t_key] = max(y_max_scales[t_key], res['skyline']['ss'].max())

    for t_key in target_keys:
        ylim = y_max_scales[t_key] * 1.15  # ← * 100 を削除
        for year in years:
            if all_res[t_key][year] is None: continue
            res = all_res[t_key][year]
            for lang in RUN_CONFIG['languages']['skyline_xray']:
                for style in RUN_CONFIG['styles']['skyline_xray']:
                    plot_structural_xray(res['skyline'], res['leakage'], t_key, year, ylim, lang, style)

# 実行
run_xray_pipeline()

In [ ]:
# =========================================================
# GIFアニメーション生成 セクション（サイズ自動調整）
# =========================================================

def create_evolution_gif(target_key, chart_type='skyline', lang='ja', style='color', duration=1.5):
    """
    指定された種類・言語・スタイルの画像を年次順に結合してGIFを作成する
    (画像サイズが異なる場合は、最初の画像のサイズにリサイズして統一する)
    """
    source_dir = os.path.join(SAVE_DIR, chart_type)
    if not os.path.exists(source_dir):
        return

    # ファイル検索パターン
    pattern = f"{chart_type}_{target_key}_*_{lang}_{style}.png"
    search_path = os.path.join(source_dir, pattern)

    files = sorted(glob.glob(search_path))
    if len(files) < 2:
        return

    # 画像の読み込みとサイズ統一
    images = []

    # 基準サイズを決定（1枚目の画像に合わせる）
    try:
        first_img = Image.open(files[0])
        base_size = first_img.size  # (width, height)
        images.append(np.array(first_img)) # 1枚目はそのまま追加

        for filename in files[1:]:
            img = Image.open(filename)

            # サイズが異なる場合のみリサイズ
            if img.size != base_size:
                img = img.resize(base_size, Image.Resampling.LANCZOS)

            images.append(np.array(img))

    except Exception as e:
        print(f"Error processing images for {target_key} {chart_type}: {e}")
        return

    # GIF保存
    if images:
        save_dir = os.path.join(SAVE_DIR, 'gif')
        os.makedirs(save_dir, exist_ok=True)

        output_name = f"evolution_{chart_type}_{target_key}_{lang}_{style}.gif"
        output_path = os.path.join(save_dir, output_name)

        try:
            imageio.mimsave(output_path, images, duration=duration, loop=0)
            print(f"  Generated GIF: {output_name}")
        except Exception as e:
             print(f"  Failed to save GIF {output_name}: {e}")

# --- 実行パイプライン ---
def run_gif_generation_pipeline():
    print(">>> Starting GIF Generation (with auto-resize)...")

    chart_types = ['skyline', 'xray']
    regions = ['shikoku', 'kanto']

    for r in regions:
        for c_type in chart_types:
            for lang in RUN_CONFIG['languages']['skyline_xray']:
                for style in RUN_CONFIG['styles']['skyline_xray']:
                    create_evolution_gif(r, chart_type=c_type, lang=lang, style=style, duration=1500)
# 実行
run_gif_generation_pipeline()

## 長期推移分析

In [ ]:
# =========================================================
# 長期推移分析 セクション
# =========================================================

# --- 1. 出力先ディレクトリの追加構築 ---
output_res_dir = os.path.join(SAVE_DIR, 'longterm')
macro_save_dir = os.path.join(output_res_dir, 'macro_analysis')
tables_dir = os.path.join(SAVE_DIR, 'tables')
for d in [output_res_dir, macro_save_dir, tables_dir]:
    os.makedirs(d, exist_ok=True)

# --- 2. グラフ描画用スタイル定義 ---
BW_LINE_STYLES = ['-', '--', '-.', ':', (0, (3, 5, 1, 5)), (0, (1, 1))]
BW_MARKERS = ['o', 's', '^', 'D', 'v', '<', '>', 'p']

def get_plot_style(idx, mode):
    if mode == 'color':
        return {'color': plt.cm.tab10(idx % 10), 'linestyle': '-', 'marker': BW_MARKERS[idx % 8]}
    else:
        return {'color': 'black', 'linestyle': BW_LINE_STYLES[idx % 6], 'marker': BW_MARKERS[idx % 8]}

# --- 3. 統合解析実行関数 ---
def run_full_integrated_analysis_v3(target_key='shikoku'):
    cfg = TARGET_CONFIG[target_key]
    years = sorted(YEAR_FILES.keys())
    leak_cols = ['g1', 'g2', 'others', 'overseas']

    all_ss_records, all_leak_records, macro_records = [], [], []

    print(f"\n>>> Processing All Data & Charts for {target_key}...")

    for year in years:
        filepath = YEAR_FILES[year]
        if not os.path.exists(filepath):
            print(f"Skipping {year}: File not found.")
            continue

        res = get_structure(year, target_key)

        df_ss = res['skyline'].copy()
        df_ss['year'] = year
        all_ss_records.append(df_ss)

        df_leak = res['leakage'].copy()
        total_ind = df_leak[['intra'] + leak_cols].sum(axis=1)
        df_leak['local_ratio'] = df_leak['intra'] / total_ind
        df_leak['net_balance'] = (df_ss['ss'] - 1.0) - (df_leak[leak_cols].sum(axis=1))
        df_leak['year'] = year
        all_leak_records.append(df_leak)

        # 全産業の集計
        t_share_all = df_ss['share'].sum()
        avg_ss_all = (df_ss['share'] * df_ss['ss']).sum() / t_share_all if t_share_all > 0 else 0
        leak_sums_all = df_leak[leak_cols].sum(axis=0)
        intra_sum_all = df_leak['intra'].sum()
        net_bal_all = (df_leak['net_balance'] * df_ss['share']).sum() / t_share_all if t_share_all > 0 else 0

        rec_all = {
            'year': year, 'group_ja': '全産業', 'group_en': MACRO_EN['全産業'],
            'ss_rate': avg_ss_all, 'net_balance': net_bal_all, 'share': t_share_all,
            'intra': intra_sum_all
        }
        for col in leak_cols:
            rec_all[f'leak_{col}'] = leak_sums_all[col]
        macro_records.append(rec_all)

        # 大分類別の集計
        for g_name, s_list in MACRO_GROUPS.items():
            g_ss = df_ss[df_ss['sector'].isin(s_list)]
            g_leak = df_leak[df_leak['sector'].isin(s_list)]
            t_share = g_ss['share'].sum()

            avg_ss = (g_ss['share'] * g_ss['ss']).sum() / t_share if t_share > 0 else 0
            leak_sums = g_leak[leak_cols].sum(axis=0)
            intra_sum = g_leak['intra'].sum()
            net_bal = (g_leak['net_balance'] * g_ss['share']).sum() / t_share if t_share > 0 else 0

            rec = {
                'year': year, 'group_ja': g_name, 'group_en': MACRO_EN[g_name],
                'ss_rate': avg_ss, 'net_balance': net_bal, 'share': t_share,
                'intra': intra_sum
            }
            for col in leak_cols:
                rec[f'leak_{col}'] = leak_sums[col]
            macro_records.append(rec)

    full_ss_df = pd.concat(all_ss_records)
    full_leak_df = pd.concat(all_leak_records)
    full_macro_df = pd.DataFrame(macro_records)
    sectors_to_watch = ['化学製品', '電気機械', '自動車', '商業', '農林水産業']

    full_ss_df.to_csv(f"{tables_dir}/{target_key}_full_ss_data.csv", index=False, encoding='utf-8-sig')
    full_leak_df.to_csv(f"{tables_dir}/{target_key}_full_leakage_data.csv", index=False, encoding='utf-8-sig')
    full_macro_df.to_csv(f"{tables_dir}/{target_key}_macro_data.csv", index=False, encoding='utf-8-sig')

    # --- 4. 可視化プロセス ---
    for lang in RUN_CONFIG['languages']['longterm']:
        t_name = cfg[f'name_{lang}']
        l_groups_full = cfg['leakage_groups'][lang]
        y_label_total = "総誘発生産に占める比率 (%)" if lang == 'ja' else "Share of total induced production (%)"
        y_label_leak = "総漏出額に占める比率 (%)" if lang == 'ja' else "Share of total leakage (%)"

        for style in RUN_CONFIG['styles']['longterm']:
            suffix = f"_{style}"
            p = PALETTES[style]

            # Chart 1. 主要部門の自給率推移
            plt.figure(figsize=(12, 7))
            for i, sector in enumerate(sectors_to_watch):
                data = full_ss_df[full_ss_df['sector'] == sector]
                label = SECTOR_EN.get(sector, sector) if lang == 'en' else sector
                # ssの * 100 を削除
                plt.plot(data['year'], data['ss'], label=label, lw=3, **get_plot_style(i, style), markersize=10)
                plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.1f}')) # ← 小数表示に変更
            plt.axhline(1.0, color='black', ls='-', alpha=0.3) # ← 1.0 に変更
            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(f"{output_res_dir}/{target_key}_ss_transition_{lang}{suffix}.png", dpi=300)
            plt.close()

            # Chart 2. 主要部門の産業収支推移
            plt.figure(figsize=(12, 7))
            for i, sector in enumerate(sectors_to_watch):
                data = full_leak_df[full_leak_df['sector'] == sector]
                label = SECTOR_EN.get(sector, sector) if lang == 'en' else sector
                plt.plot(data['year'], data['net_balance'], label=label, lw=3, **get_plot_style(i, style), markersize=10)
            plt.axhline(0, color='black', ls='-', alpha=0.3)
            # plt.title(f"{t_name} Net Balance" if lang=='en' else f"{t_name} 産業収支比率の推移", pad=20)
            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(f"{output_res_dir}/{target_key}_balance_transition_{lang}{suffix}.png", dpi=300)
            plt.close()

            # Chart 3. マクロ自給率推移
            plt.figure(figsize=(12, 7))
            unique_groups = full_macro_df[f'group_{lang}'].unique()
            for i, g in enumerate(unique_groups):
                data = full_macro_df[full_macro_df[f'group_{lang}'] == g]
                is_total = (g == '全産業' or g == 'All Industries')
                # ss_rateの * 100 を削除
                plt.plot(data['year'], data['ss_rate'], label=g,
                         lw=6 if is_total else 3.5, alpha=1.0 if is_total else 0.7,
                         **get_plot_style(i, style), markersize=12 if is_total else 9)
            plt.axhline(1.0, color='black', ls='--') # ← 1.0 に変更
            plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.1f}')) # ← 小数表示に変更

            # y軸ラベルの変更（(%) を削除）
            y_label_ss = "自給率" if lang == 'ja' else "Self-sufficiency Rate"
            plt.ylabel(y_label_ss, fontsize=16)

            plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(f"{macro_save_dir}/{target_key}_macro_ss_{lang}{suffix}.png", dpi=300)
            plt.close()

            # Chart 4. 域内循環構造グラフ (Figure 9)
            groups_to_plot = ['全産業']
            for g_ja in groups_to_plot:
                if g_ja not in full_macro_df['group_ja'].unique(): continue

                df_g = full_macro_df[full_macro_df['group_ja'] == g_ja].sort_values('year')
                g_display = MACRO_EN[g_ja] if lang == 'en' else g_ja
                years_str = df_g['year'].astype(str).values
                safe_g = g_ja.replace('/', '_')

                fig, ax = plt.subplots(figsize=(12, 8))

                val_intra = df_g['intra'].values
                val_leaks = df_g[[f'leak_{c}' for c in leak_cols]].values
                total_induced = val_intra + val_leaks.sum(axis=1)

                bottom_pct = np.zeros(len(df_g))

                # 域内の描画
                pct_intra = (val_intra / total_induced) * 100
                ax.bar(years_str, pct_intra, bottom=bottom_pct, label=l_groups_full[0],
                       fc=p['leakage'][0], ec='black', hatch=p['hatches'][0], alpha=0.85)

                for j, val in enumerate(pct_intra):
                    if val > 6.0:
                        ax.text(j, bottom_pct[j] + val/2, f'{val:.1f}%',
                                ha='center', va='center', fontweight='bold', fontsize=15, color='black',
                                bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))
                bottom_pct += pct_intra

                # 流出分の描画
                for i, col in enumerate(leak_cols):
                    vals = df_g[f'leak_{col}'].values
                    pct_vals = (vals / total_induced) * 100

                    ax.bar(years_str, pct_vals, bottom=bottom_pct, label=l_groups_full[i+1],
                           fc=p['leakage'][i+1], ec='black', hatch=p['hatches'][i+1], alpha=0.85)

                    for j, val in enumerate(pct_vals):
                        if val > 4.0:
                            ax.text(j, bottom_pct[j] + val/2, f'{val:.1f}%',
                                     ha='center', va='center', fontweight='bold', fontsize=15,
                                     bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))
                    bottom_pct += pct_vals

                title_text = f"{t_name} {g_display}：地域経済循環の構造 (対 全生産誘発額)" if lang=='ja' else f"{t_name} {g_display}: Structure of Regional Economic Circulation"
                # ax.set_title(title_text, fontweight='bold', pad=20)
                ax.set_ylabel(y_label_total , labelpad=10)
                ax.set_ylim(0, 100)

                handles, labels_legend = ax.get_legend_handles_labels()
                ax.legend(handles[::-1], labels_legend[::-1], bbox_to_anchor=(1.02, 1), loc='upper left', title="Flow Destination")
                ax.grid(axis='y', linestyle='--', alpha=0.3)

                plt.tight_layout()
                save_path = os.path.join(macro_save_dir, f"{target_key}_total_circulation_structure_{safe_g}_{lang}{suffix}.png")
                plt.savefig(save_path, dpi=300, bbox_inches='tight')
                plt.close()

                # --- 追加箇所: Chart 5. 域外漏出の内訳グラフ (Figure 10相当) ---
                if g_ja == '全産業': # 全産業のときのみ作成
                    fig, ax = plt.subplots(figsize=(10, 6))

                    # 漏出額だけを取り出して100%積み上げ
                    leak_vals = df_g[[f'leak_{c}' for c in leak_cols]].values
                    leak_total = leak_vals.sum(axis=1)
                    leak_pct = np.divide(leak_vals.T, leak_total, where=leak_total!=0).T * 100

                    bottom_pct = np.zeros(len(df_g))

                    for i, col in enumerate(leak_cols):
                        # PALETTESのindex: 0は域内なので、1以降(i+1)を使う
                        idx = i + 1
                        vals = leak_pct[:, i]

                        ax.bar(years_str, vals, bottom=bottom_pct,
                               label=l_groups_full[idx],
                               fc=p['leakage'][idx], ec='black', hatch=p['hatches'][idx], alpha=0.85)

                        # 数値ラベル
                        for j, val in enumerate(vals):
                            if val > 5.0: # 5%以上なら表示
                                ax.text(j, bottom_pct[j] + val/2, f'{val:.1f}%',
                                        ha='center', va='center', fontweight='bold', fontsize=12, color='black',
                                        bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))
                        bottom_pct += vals

                    title_text = f"{t_name} Breakdown of Leakage Destinations" if lang=='en' else f"{t_name} 域外漏出先の内訳構成比"
                    # ax.set_title(title_text, fontweight='bold', pad=20)
                    ax.set_ylabel(y_label_leak, labelpad=10)
                    ax.set_ylim(0, 100)

                    handles, labels_legend = ax.get_legend_handles_labels()
                    ax.legend(handles[::-1], labels_legend[::-1], bbox_to_anchor=(1.02, 1), loc='upper left')
                    ax.grid(axis='y', linestyle='--', alpha=0.3)

                    plt.tight_layout()
                    # ファイル名: shikoku_leakage_ratio_only_全産業_en_color.png となる
                    save_path = os.path.join(macro_save_dir, f"{target_key}_leakage_ratio_only_{safe_g}_{lang}{suffix}.png")
                    plt.savefig(save_path, dpi=300, bbox_inches='tight')
                    plt.close()
                # --- 追加終了 ---

# --- 5. 比較指標の計算・描画 ---
def calculate_all_regional_ssr():
    ssr_records = []
    for r_key in ALL_REGIONS:
        for year in sorted(YEAR_FILES.keys()):
            if not os.path.exists(YEAR_FILES[year]):
                continue
            res = get_structure(year, r_key)
            sky_df = res['skyline']
            avg_ssr = (sky_df['share'] * sky_df['ss']).sum()

            ssr_records.append({'Year': year, 'Region': r_key, 'SSR': avg_ssr})

    return pd.DataFrame(ssr_records)

def plot_regional_ssr_comparison(df_ssr, lang='ja', style='color'):
    t = TRANSLATIONS[lang]
    suffix = f"_{style}"
    save_dir = os.path.join(SAVE_DIR, "interregional_comparison")
    os.makedirs(save_dir, exist_ok=True)

    plt.figure(figsize=(12, 7))
    color_palette = sns.color_palette("husl", len(ALL_REGIONS))

    for i, r in enumerate(ALL_REGIONS):
        data = df_ssr[df_ssr['Region'] == r]
        is_shikoku = (r == 'shikoku')

        if style == 'color':
            line_cfg = {'color': color_palette[i], 'marker': BW_STYLES[r]['marker'], 'ls': '-'}
        else:
            s = BW_STYLES[r]
            line_cfg = {'color': s['color'], 'marker': s['marker'], 'ls': s['ls']}

        # SSRの * 100 を削除
        plt.plot(data['Year'], data['SSR'], label=t['regions'][r],
                 linewidth=4 if is_shikoku else 1.5, alpha=1.0 if is_shikoku else 0.7,
                 zorder=10 if is_shikoku else 5, **line_cfg)

    # 1.0 に変更
    plt.axhline(1.0, color='black', linestyle='--', alpha=0.5, label='Self-sufficiency = 1.0')

    plt.ylabel("Total SSR", fontsize=16)  # ← (%) を削除
    plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.1f}'))  # ← 小数表示に変更
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=12)
    plt.grid(True, ls=':', alpha=0.6)

    save_path = os.path.join(save_dir, f"compare_total_ssr_{lang}{suffix}.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")

# =========================================================
# 実行ブロック
# =========================================================
for t_key in ['shikoku', 'kanto']:
    run_full_integrated_analysis_v3(t_key)

print("\n>>> Calculating regional SSR for comparison...")
df_ssr = calculate_all_regional_ssr()
for lang in RUN_CONFIG['languages']['longterm']:
    for style in RUN_CONFIG['styles']['longterm']:
        plot_regional_ssr_comparison(df_ssr, lang=lang, style=style)

print("\n[完了] 視認性を向上させた長期推移分析および構造解剖グラフの生成が終わりました。")

In [ ]:
# =========================================================
# 論文本文の主要数値を再現・検証するセクション
# =========================================================
def verify_main_text_numbers():
    print("\n" + "="*70)
    print("【検証】本文の主要数値の再現チェック")
    print("="*70)

    yearly = {}
    for year in ['1985', '1990', '1995', '2005']:
        yearly[year] = get_structure(year, 'shikoku')['skyline'].copy()

    # 1) Aggregate SSR
    print("\n[1] Aggregate SSR")
    for year in ['1985', '1990', '1995', '2005']:
        val = calc_weighted_ssr(yearly[year])
        print(f"  {year}: {val:.4f}")

    # 2) CV
    print("\n[2] Skyline CV")
    for year in ['1985', '1990', '1995', '2005']:
        val = calc_cv_ssr(yearly[year])
        print(f"  {year}: {val:.4f}")

    # 3) Export base share
    print("\n[3] Export-base share (SSR > 1.0)")
    for year in ['1985', '1990', '1995', '2005']:
        val = calc_export_base_share(yearly[year]) * 100
        print(f"  {year}: {val:.2f}%")

    # 4) Sharpening Index
    print("\n[4] Sharpening Index (1985→2005)")
    phi_df = calc_sharpening_index(yearly['1985'], yearly['2005'])

    sectors_to_check = [
        '非鉄金属', '化学製品', 'パルプ・紙・板紙・加工紙',
        '電気機械', '繊維工業製品', 'その他輸送用機械'
    ]
    for s in sectors_to_check:
        row = phi_df[phi_df['sector'] == s]
        if len(row) == 0:
            print(f"  {s}: not found")
            continue
        r = row.iloc[0]
        print(f"  {s}: phi={r['phi']:.4f}, share {r['share_t0']:.4f}->{r['share_t1']:.4f}, ss {r['ss_t0']:.4f}->{r['ss_t1']:.4f}")

    return yearly, phi_df

yearly_sky, phi_check_df = verify_main_text_numbers()

def verify_quadrant_consistency(phi_df):
    print("\n" + "="*70)
    print("【検証】Sharpening Index と四象限分類の整合")
    print("="*70)

    def classify(row):
        if row['d_share'] < 0 and row['d_ss'] > 0:
            return 'Q2 Structural Sharpening'
        elif row['d_share'] < 0 and row['d_ss'] < 0:
            return 'Q3 Double Decline'
        elif row['d_share'] > 0 and row['d_ss'] > 0:
            return 'Q1 Growth Driver'
        elif row['d_share'] > 0 and row['d_ss'] < 0:
            return 'Q4 Hollow Growth'
        else:
            return 'Boundary/Zero'

    chk = phi_df.copy()
    chk['quadrant'] = chk.apply(classify, axis=1)

    for s in ['非鉄金属', '化学製品', 'パルプ・紙・板紙・加工紙', '電気機械', '繊維工業製品', 'その他輸送用機械']:
        row = chk[chk['sector'] == s]
        if len(row) == 0:
            continue
        r = row.iloc[0]
        print(f"{s:<20} | quadrant={r['quadrant']:<28} | phi={r['phi']:.4f}")

verify_quadrant_consistency(phi_check_df)



def export_validation_tables(yearly_sky, phi_df):
    out_dir = os.path.join(SAVE_DIR, 'tables')
    os.makedirs(out_dir, exist_ok=True)

    # 年別 summary
    summary_rows = []
    for year, df in yearly_sky.items():
        summary_rows.append({
            'year': year,
            'aggregate_ssr': calc_weighted_ssr(df),
            'cv_ssr': calc_cv_ssr(df),
            'export_base_share_pct_ssr_gt_1': calc_export_base_share(df) * 100
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(os.path.join(out_dir, 'validation_main_text_summary.csv'),
                      index=False, encoding='utf-8-sig')

    # Sharpening
    phi_df.to_csv(os.path.join(out_dir, 'validation_sharpening_index_1985_2005.csv'),
                  index=False, encoding='utf-8-sig')

    print("\n検証テーブルを保存しました。")
    display(summary_df)

def compare_export_base_definitions(sky_df):
    out = {}

    out['SSR > 1.0'] = sky_df.loc[sky_df['ss'] > 1.0, 'share'].sum() * 100
    out['SSR >= 1.0'] = sky_df.loc[sky_df['ss'] >= 1.0, 'share'].sum() * 100

    # 製造業＋農林水産＋鉱業くらいの広義 tradables
    tradable_like = [
        '農林水産業', '鉱業', '飲食料品', '繊維工業製品', '製材・木製品・家具',
        'パルプ・紙・板紙・加工紙', '出版・印刷', '化学製品', '石油・石炭製品',
        'プラスチック製品', '窯業・土石製品', '鉄鋼', '非鉄金属', '金属製品',
        '一般機械', '電気機械', '自動車', 'その他輸送用機械', '精密機械',
        'その他の製造工業製品'
    ]

    out['Tradables only & SSR > 1.0'] = sky_df.loc[
        (sky_df['sector'].isin(tradable_like)) & (sky_df['ss'] > 1.0),
        'share'
    ].sum() * 100

    out['Tradables only & SSR >= 1.0'] = sky_df.loc[
        (sky_df['sector'].isin(tradable_like)) & (sky_df['ss'] >= 1.0),
        'share'
    ].sum() * 100

    return pd.Series(out)



export_validation_tables(yearly_sky, phi_check_df)

for year in ['1985', '2005']:
    print(f"\n=== {year} ===")
    print(compare_export_base_definitions(yearly_sky[year]).round(3))

## 支店経済化分析

In [ ]:
# =========================================================
# 支店経済化分析 セクション
# =========================================================

PLOT_REGIONS = [r for r in ALL_REGIONS if r != 'kanto']
BW_MARKERS = ['o', 's', '^', 'D', 'v', '<', '>', 'p']

# --- 1. 計算コア ---
def calculate_all_regional_metrics():
    service_sectors = ['金融・保険', '情報通信', '広告・対事業所サービス']
    dep_list, idx_list = [], []
    num_sec = 35

    for year in sorted(YEAR_FILES.keys()):
        if not os.path.exists(YEAR_FILES[year]):
            continue
        print(f">>> Processing Year: {year}")

        Z, X, F_list, M, names = get_year_data(year)

        kanto_idx = TARGET_CONFIG['kanto']['idx']
        kanto_service_idx = [np.where(names == s)[0][0] + kanto_idx * num_sec for s in service_sectors]

        n = len(X)
        X_safe = np.where(X == 0, 1e-9, X)
        A = Z @ np.diag(1 / X_safe)
        Total_D = (A @ X) + np.sum(F_list, axis=0)
        m = np.clip(M / np.where(Total_D <= 0, 1, Total_D), 0, 1)
        L_dom = np.linalg.inv(np.eye(n) - (np.diag(1 - m) @ A))

        for r_key in ALL_REGIONS:
            t_conf = TARGET_CONFIG[r_key]
            region_slice = slice(t_conf['idx'] * num_sec, (t_conf['idx'] + 1) * num_sec)

            X_reg = X[region_slice]
            X_reg_sum = X_reg.sum()

            L_from_kanto = L_dom[kanto_service_idx, region_slice].sum(axis=0)
            valid_idx = X_reg > 0

            deps = L_from_kanto[valid_idx] * 100
            valid_names = names[valid_idx]
            valid_X = X_reg[valid_idx]

            for i in range(len(deps)):
                dep_list.append({
                    'Year': year,
                    'Region': r_key,
                    'Industry': valid_names[i],
                    'Dependency': deps[i],
                    'Output': valid_X[i],
                    'Output_Share_Region': valid_X[i] / X_reg_sum if X_reg_sum > 0 else np.nan
                })

            w_avg = (np.sum(L_from_kanto[valid_idx] * X_reg[valid_idx]) / X_reg_sum * 100) if X_reg_sum > 0 else 0

            res = analyze_structure(Z, X, F_list, M, names, r_key)
            ss_vals = res['skyline']['ss'].values

            mean_ss = np.mean(ss_vals)
            cv = np.std(ss_vals) / mean_ss if mean_ss != 0 else 0

            p = X_reg / X_reg_sum if X_reg_sum > 0 else np.zeros_like(X_reg)
            entropy = -np.sum(p[p > 0] * np.log(p[p > 0]))
            avg_ssr = (res['skyline']['share'] * res['skyline']['ss']).sum()

            idx_list.append({
                'Year': year,
                'Region': r_key,
                'Skyline_CV': cv,
                'Diversity': entropy,
                'Weighted_Avg': w_avg,
                'SSR': avg_ssr
            })

    return pd.DataFrame(dep_list), pd.DataFrame(idx_list)

# --- 2. 可視化エンジン ---
def finalize_plot(title, ylabel, filename, comp_dir, lang, show_xlabel=True):
    # plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel(ylabel)

    # 条件分岐の内側でのみラベルを描画するようにし、重複を削除します
    if show_xlabel:
        plt.xlabel(TRANSLATIONS[lang]['years'])

    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, ls=':', alpha=0.6)
    save_path = os.path.join(comp_dir, filename)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_styled_branch_economy_graphics(full_dep, full_idx, output_root, lang='ja', style='color'):
    t = TRANSLATIONS[lang]
    suffix = f"_{style}"

    dirs = {
        'trends': os.path.join(output_root, "dependency_analysis"),
        'struct': os.path.join(output_root, "cv_and_entropy"),
        'detail': os.path.join(output_root, "sector_details"),
        'robust': os.path.join(output_root, "shikoku_dependency"),
        'additional': os.path.join(output_root, "additional")
    }
    for d in dirs.values(): os.makedirs(d, exist_ok=True)

    color_palette = sns.color_palette("husl", len(ALL_REGIONS))

    def get_line_cfg(r_idx, r_key):
        if style == 'color':
            return {'color': color_palette[r_idx], 'marker': BW_STYLES[r_key]['marker'], 'ls': '-'}
        else:
            s = BW_STYLES[r_key]
            return {'color': s['color'], 'marker': s['marker'], 'ls': s['ls']}

    # (1) Dependency Trend
    plt.figure(figsize=(10, 6))
    for i, r in enumerate(PLOT_REGIONS):
        data = full_idx[full_idx['Region'] == r]
        plt.plot(data['Year'], data['Weighted_Avg'], label=t['regions'][r],
                 linewidth=3.5 if r == 'shikoku' else 1.5, markersize=7, **get_line_cfg(i, r))
    # plt.title(t['dep_title'], fontsize=14, fontweight='bold');
    plt.ylabel(t['dep_ylabel'], fontsize=14, labelpad=10)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); plt.grid(True, ls=':', alpha=0.6)
    plt.savefig(os.path.join(dirs['trends'], f"dependency_trend_{lang}{suffix}.png"), dpi=300, bbox_inches='tight'); plt.close()

    # (2) CV & Entropy
    for col, title, ylabel, fname in [('Skyline_CV', t['cv_title'], t['cv_ylabel'], "skyline_cv"),
                                      ('Diversity', t['ent_title'], t['ent_ylabel'], "entropy")]:
        plt.figure(figsize=(10, 6))
        for i, r in enumerate(ALL_REGIONS):
            data = full_idx[full_idx['Region'] == r]
            plt.plot(data['Year'], data[col], label=t['regions'][r],
                     linewidth=3.5 if r in ['shikoku', 'kanto'] else 1.2, **get_line_cfg(i, r))
        # plt.title(title, fontsize=14, fontweight='bold'); plt.ylabel(ylabel)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); plt.grid(True, ls=':', alpha=0.6)
        plt.savefig(os.path.join(dirs['struct'], f"{fname}_{lang}{suffix}.png"), dpi=300, bbox_inches='tight'); plt.close()

    # (2.5) Shikoku specific dual-axis
    sh_idx = full_idx[full_idx['Region'] == 'shikoku']
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax2 = ax1.twinx()
    c1, c2 = ('blue', 'green') if style == 'color' else ('black', '#666666')
    ax1.plot(sh_idx['Year'], sh_idx['Skyline_CV'], color=c1, marker='s', label=t['cv_label'])
    ax2.plot(sh_idx['Year'], sh_idx['Diversity'], color=c2, marker='^', ls='--', label=t['entropy_label'])
    # ax1.set_xlabel(t['years']);
    ax1.set_ylabel(t['cv_label'], color=c1); ax2.set_ylabel(t['entropy_label'], color=c2)
    # plt.title(t['idx_title'], fontsize=14, fontweight='bold')
    lines1, labels1 = ax1.get_legend_handles_labels(); lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left'); plt.grid(True, ls=':', alpha=0.6)
    plt.savefig(os.path.join(dirs['additional'], f"shikoku_structural_indices_{lang}{suffix}.png"), dpi=300, bbox_inches='tight'); plt.close()

    # (3) Heatmap
    latest_year = full_dep['Year'].max()
    latest_data = full_dep[full_dep['Year'] == latest_year]
    plt.figure(figsize=(12, 11))
    plot_data = latest_data[latest_data['Region'] != 'kanto']

    pivot_df = plot_data.pivot(index='Industry', columns='Region', values='Dependency')
    pivot_df = pivot_df.reindex(index=plot_data['Industry'].unique(), columns=PLOT_REGIONS)

    if lang == 'en':
        pivot_df.index = [SECTOR_EN.get(ind, ind) for ind in pivot_df.index]
    pivot_df.columns = [t['regions'].get(reg, reg) for reg in pivot_df.columns]

    sns.heatmap(pivot_df, cmap='YlGnBu' if style == 'color' else 'Greys', annot=False, linewidths=0.1, linecolor='#EEEEEE', cbar_kws={'label': t['dep_ylabel']})
    # plt.title(f"{latest_year} {t['heatmap_title']}", fontsize=14, fontweight='bold')
    plt.xlabel(t['heatmap_xlabel']); plt.ylabel(t['heatmap_ylabel'])
    plt.savefig(os.path.join(dirs['detail'], f"dep_heatmap_{lang}{suffix}.png"), dpi=300, bbox_inches='tight'); plt.close()

    # (4) Shikoku key sectors
    plt.figure(figsize=(10, 6))
    shikoku_dep = full_dep[full_dep['Region'] == 'shikoku']
    BW_LINE_TYPES = ['-', '--', '-.', ':']

    for i, (ind_ja, label_name) in enumerate(t['ind_map'].items()):
        data = shikoku_dep[shikoku_dep['Industry'] == ind_ja]
        if style == 'bw':
            plt.plot(data['Year'], data['Dependency'], label=label_name, color='black',
                     ls=BW_LINE_TYPES[i % len(BW_LINE_TYPES)], marker=BW_MARKERS[i % len(BW_MARKERS)], markersize=8, alpha=0.7, lw=2)
        else:
            plt.plot(data['Year'], data['Dependency'], label=label_name, alpha=0.8, ls='-', marker='o', lw=2)

    plt.plot(sh_idx['Year'], sh_idx['Weighted_Avg'], color='red' if style=='color' else 'black',
             linewidth=4, marker='D', markersize=10, label=t['avg_label'], zorder=20)
    # plt.title(t['dep_shikoku_title'], fontsize=14, fontweight='bold'))
    plt.ylabel(t['dep_ylabel'], fontsize=14, labelpad=10)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); plt.grid(True, ls=':', alpha=0.6)
    plt.savefig(os.path.join(dirs['robust'], f"shikoku_key_sectors_dep_{lang}{suffix}.png"), dpi=300, bbox_inches='tight'); plt.close()

    # (5) Dependency boxplot
    plt.figure(figsize=(12, 7))
    dist_df = full_dep[full_dep['Region'] != 'kanto'].copy()
    dist_df['Region_Name'] = dist_df['Region'].map(lambda x: t['regions'].get(x, x))
    sns.boxplot(data=dist_df, x='Region_Name', y='Dependency', palette='Set3' if style == 'color' else None, color='white' if style == 'bw' else None)
    plt.yscale('log')
    # plt.title(f"{latest_year} {t['dist_title']}", fontsize=14, fontweight='bold')
    plt.grid(True, which="both", ls=":", alpha=0.5)
    plt.savefig(os.path.join(dirs['additional'], f"dep_box_{lang}{suffix}.png"), dpi=300, bbox_inches='tight'); plt.close()

def plot_comparison_styled(all_idx, output_root, lang='ja', style='color'):
    t = TRANSLATIONS[lang]
    suffix = f"_{style}"
    comp_dir = os.path.join(output_root, "comparison")
    os.makedirs(comp_dir, exist_ok=True)

    def get_line_cfg(r_idx, r_key):
        if style == 'color':
            return {'color': sns.color_palette("husl", len(ALL_REGIONS))[r_idx], 'marker': BW_STYLES[r_key]['marker'], 'ls': '-'}
        else:
            return {'color': BW_STYLES[r_key]['color'], 'marker': BW_STYLES[r_key]['marker'], 'ls': BW_STYLES[r_key]['ls']}

    # (1) Weighted average
    plt.figure(figsize=(10, 6))
    for i, r in enumerate(PLOT_REGIONS):
        data = all_idx[all_idx['Region'] == r]
        plt.plot(data['Year'], data['Weighted_Avg'], label=t['regions'][r],
                 linewidth=3.5 if r == 'shikoku' else 1.5, zorder=10 if r == 'shikoku' else 5, **get_line_cfg(i, r))
    # plt.title(t['dep_title'], fontsize=14, fontweight='bold')
    # plt.xlabel(t['years']))
    plt.ylabel(t['dep_ylabel'], fontsize=14, labelpad=10)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left'); plt.grid(True, ls=':', alpha=0.6)
    plt.savefig(os.path.join(comp_dir, f"compare_dep_avg_{lang}{suffix}.png"), dpi=300, bbox_inches='tight'); plt.close()

# (2) Skyline CV
    plt.figure(figsize=(10, 6))
    for i, r in enumerate(ALL_REGIONS):
        data = all_idx[all_idx['Region'] == r]
        cfg = get_line_cfg(i, r)
        if style == 'bw' and r == 'kanto': cfg.update({'color': '#000000', 'ls': ':'})
        plt.plot(data['Year'], data['Skyline_CV'], label=t['regions'][r],
                 linewidth=3.5 if r in ['kanto', 'shikoku'] else 1.2, zorder=15 if r == 'shikoku' else 5, **cfg)
    finalize_plot(t['cv_title'], t['cv_ylabel'], f"compare_cv_{lang}{suffix}.png", comp_dir, lang, show_xlabel=False)

    # (3) Diversity
    plt.figure(figsize=(10, 6))
    for i, r in enumerate(ALL_REGIONS):
        data = all_idx[all_idx['Region'] == r]
        cfg = get_line_cfg(i, r)
        if style == 'bw' and r == 'kanto': cfg.update({'color': '#000000', 'ls': ':'})
        plt.plot(data['Year'], data['Diversity'], label=t['regions'][r],
                 linewidth=3.5 if r in ['kanto', 'shikoku'] else 1.2, **cfg)
    finalize_plot(t['ent_title'], t['ent_ylabel'], f"compare_entropy_{lang}{suffix}.png", comp_dir, lang, show_xlabel=False)

def plot_shikoku_macro_dependency_with_avg(full_dep, full_idx, output_root, lang='ja', style='color'):
    t = TRANSLATIONS[lang]
    suffix = f"_{style}"
    save_dir = os.path.join(output_root, "macro_analysis")
    os.makedirs(save_dir, exist_ok=True)

    ind_to_macro = {ind: macro for macro, industries in MACRO_GROUPS.items() for ind in industries}
    sh_dep = full_dep[full_dep['Region'] == 'shikoku'].copy()
    sh_dep['Macro_Group'] = sh_dep['Industry'].map(ind_to_macro)
    sh_dep = sh_dep.dropna(subset=['Macro_Group']).copy()

    sh_idx = full_idx[full_idx['Region'] == 'shikoku'].copy()

    # 各年・各マクロ部門ごとの output-weighted average dependency
    macro_trend = (
        sh_dep.groupby(['Year', 'Macro_Group'])
        .apply(lambda g: np.average(g['Dependency'], weights=g['Output']) if g['Output'].sum() > 0 else np.nan)
        .reset_index(name='Dependency')
    )

    plt.figure(figsize=(12, 8))
    bw_markers, bw_lines = ['o', 's', 'v', '^'], ['--', '-.', ':', '--']

    for i, m_ja in enumerate(MACRO_GROUPS.keys()):
        data = macro_trend[macro_trend['Macro_Group'] == m_ja]
        label_name = MACRO_EN[m_ja] if lang == 'en' else m_ja
        if style == 'color':
            plt.plot(
                data['Year'], data['Dependency'],
                label=label_name, marker='o', linewidth=2.5, alpha=0.6
            )
        else:
            plt.plot(
                data['Year'], data['Dependency'],
                label=label_name, color='#666666',
                marker=bw_markers[i % 4], ls=bw_lines[i % 4],
                linewidth=2, alpha=0.7
            )

    plt.plot(
        sh_idx['Year'], sh_idx['Weighted_Avg'],
        label=t['avg_label'],
        color='red' if style == 'color' else 'black',
        linewidth=5, marker='D', markersize=12, zorder=10
    )

    plt.ylabel(t['dep_ylabel'], fontsize=16, labelpad=10)
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=14, frameon=True)
    plt.grid(True, ls=':', alpha=0.6)
    plt.savefig(
        os.path.join(save_dir, f"shikoku_macro_dep_with_avg_{lang}{suffix}.png"),
        dpi=300, bbox_inches='tight'
    )
    plt.close()


# --- 3. 実行パイプライン ---
def run_unified_branch_economy_pipeline_v3():
    print(">>> Calculating dependency and structural metrics...")
    full_dep, full_idx = calculate_all_regional_metrics()

    tables_dir = os.path.join(SAVE_DIR, "tables")
    os.makedirs(tables_dir, exist_ok=True)
    full_dep.to_csv(f"{tables_dir}/raw_dependency_all.csv", index=False, encoding='utf-8-sig')
    full_idx.to_csv(f"{tables_dir}/raw_structural_indices.csv", index=False, encoding='utf-8-sig')

    for lang in RUN_CONFIG['languages']['branch_economy']:
        for style in RUN_CONFIG['styles']['branch_economy']:
            print(f">>> Generating Graphics: Language={lang}, Style={style}")
            plot_styled_branch_economy_graphics(full_dep, full_idx, SAVE_DIR, lang=lang, style=style)
            plot_comparison_styled(full_idx, SAVE_DIR, lang=lang, style=style)
            plot_shikoku_macro_dependency_with_avg(full_dep, full_idx, SAVE_DIR, lang=lang, style=style)

    print(f"支店経済化分析が完了しました。出力先: {SAVE_DIR}")
    return full_dep, full_idx

# 実行
full_dep, full_idx = run_unified_branch_economy_pipeline_v3()


ind_to_macro = {ind: macro for macro, industries in MACRO_GROUPS.items() for ind in industries}
sh_dep = full_dep[full_dep['Region'] == 'shikoku'].copy()
sh_dep['Macro_Group'] = sh_dep['Industry'].map(ind_to_macro)

macro_mean_old = (
    sh_dep.groupby(['Year', 'Macro_Group'])['Dependency']
    .mean()
    .reset_index(name='Unweighted')
)

macro_mean_new = (
    sh_dep.dropna(subset=['Macro_Group'])
    .groupby(['Year', 'Macro_Group'])
    .apply(lambda g: np.average(g['Dependency'], weights=g['Output']) if g['Output'].sum() > 0 else np.nan)
    .reset_index(name='Weighted')
)

compare_macro = macro_mean_old.merge(macro_mean_new, on=['Year', 'Macro_Group'], how='outer')
display(compare_macro.sort_values(['Macro_Group', 'Year']))

macro_weighted = (
    sh_dep.dropna(subset=['Macro_Group'])
    .groupby(['Year', 'Macro_Group'])
    .apply(lambda g: np.average(g['Dependency'], weights=g['Output']) if g['Output'].sum() > 0 else np.nan)
    .reset_index(name='Dependency')
)

for macro in MACRO_GROUPS.keys():
    d = macro_weighted[macro_weighted['Macro_Group'] == macro].sort_values('Year')
    print(f"\n[{macro}]")
    for _, row in d.iterrows():
        print(f"{int(row['Year'])}: {row['Dependency']:.3f}")

In [ ]:
# L_from_kanto の産業間相関を確認
import pandas as pd
import numpy as np

# 最初の年で診断
year = sorted(YEAR_FILES.keys())[0]
Z, X, F_list, M, names = get_year_data(year)
num_sec = 35
n = len(X)
X_safe = np.where(X == 0, 1e-9, X)
A = Z @ np.diag(1 / X_safe)
Total_D = (A @ X) + np.sum(F_list, axis=0)
m = np.clip(M / np.where(Total_D <= 0, 1, Total_D), 0, 1)
L_dom = np.linalg.inv(np.eye(n) - (np.diag(1 - m) @ A))

service_sectors = ['金融・保険', '情報通信', '広告・対事業所サービス']
kanto_idx = TARGET_CONFIG['kanto']['idx']
kanto_service_idx = [np.where(names == s)[0][0] + kanto_idx * num_sec for s in service_sectors]

# 四国スライス
shikoku_slice = slice(TARGET_CONFIG['shikoku']['idx'] * num_sec,
                      (TARGET_CONFIG['shikoku']['idx'] + 1) * num_sec)

L_cols = L_dom[kanto_service_idx, shikoku_slice]  # shape: (3, 35)

# 3業種の列和を確認
print("各関東サービス業種からの波及係数（四国産業別）:")
for i, s in enumerate(service_sectors):
    print(f"\n{s}: min={L_cols[i].min():.4f}, max={L_cols[i].max():.4f}, "
          f"std={L_cols[i].std():.4f}")

# 産業間の相関（3×3）
L_sum = L_cols  # (3, 35)
corr = np.corrcoef(L_sum)
print("\n\n関東サービス3業種間の産業ベクトル相関行列:")
print(np.round(corr, 3))

# マクロ部門間の平均値の相関（年次）
ind_to_macro = {ind: macro for macro, industries in MACRO_GROUPS.items() for ind in industries}
sh_dep = full_dep[full_dep['Region'] == 'shikoku'].copy()
sh_dep['Macro_Group'] = sh_dep['Industry'].map(ind_to_macro)
macro_trend = sh_dep.groupby(['Year', 'Macro_Group'])['Dependency'].mean().unstack()
print("\nマクロ部門間の年次相関行列:")
print(macro_trend.corr().round(3))


In [ ]:
# =========================================================
# 論文記述用：依存度の数値サマリーをprint
# =========================================================

def print_dependency_summary(full_dep, full_idx):
    print("\n" + "="*60)
    print("【Figure 12用】四国 マクロ部門別 関東プロデューサーサービス依存度")
    print("="*60)

    ind_to_macro = {ind: macro for macro, industries in MACRO_GROUPS.items() for ind in industries}
    sh_dep = full_dep[full_dep['Region'] == 'shikoku'].copy()
    sh_dep['Macro_Group'] = sh_dep['Industry'].map(ind_to_macro)

    # 図と整合するよう output-weighted average に統一
    macro_trend = (
        sh_dep.dropna(subset=['Macro_Group'])
        .groupby(['Year', 'Macro_Group'])
        .apply(lambda g: np.average(g['Dependency'], weights=g['Output']) if g['Output'].sum() > 0 else np.nan)
        .reset_index(name='Dependency')
    )

    for macro in MACRO_GROUPS.keys():
        print(f"\n  [{macro}]")
        data = macro_trend[macro_trend['Macro_Group'] == macro].sort_values('Year')
        for _, row in data.iterrows():
            print(f"    {int(row['Year'])}: {row['Dependency']:.3f}")

    print("\n" + "="*60)
    print("【Figure 12用】地域加重平均（Regional Average Weighted）")
    print("="*60)
    sh_idx = full_idx[full_idx['Region'] == 'shikoku'].sort_values('Year')
    for _, row in sh_idx.iterrows():
        print(f"  {int(row['Year'])}: {row['Weighted_Avg']:.3f}")

    print("\n" + "="*60)
    print("【Figure 11用】全地域 加重平均依存度")
    print("="*60)
    for region in PLOT_REGIONS:
        print(f"\n  [{region}]")
        data = full_idx[full_idx['Region'] == region].sort_values('Year')
        for _, row in data.iterrows():
            print(f"    {int(row['Year'])}: {row['Weighted_Avg']:.3f}")

    print("\n" + "="*60)
    print("【本文記述用】1995→2005 変化幅サマリー（四国）")
    print("="*60)

    macro_trend = macro_trend.copy()
    macro_trend['Year'] = macro_trend['Year'].astype(int)

    sh_idx = sh_idx.copy()
    sh_idx['Year'] = sh_idx['Year'].astype(int)

    for macro in MACRO_GROUPS.keys():
        data = macro_trend[macro_trend['Macro_Group'] == macro].sort_values('Year')
        val_1995 = data[data['Year'] == 1995]['Dependency'].values
        val_2005 = data[data['Year'] == 2005]['Dependency'].values
        if len(val_1995) > 0 and len(val_2005) > 0:
            delta = val_2005[0] - val_1995[0]
            ratio = val_2005[0] / val_1995[0] if val_1995[0] != 0 else np.nan
            print(f"  {macro}: {val_1995[0]:.3f} → {val_2005[0]:.3f}  (Δ{delta:+.3f}, ×{ratio:.2f})")
        else:
            print(f"  {macro}: データなし")
            print(f"    利用可能なYear: {sorted(data['Year'].unique().tolist())}")

    val_1995_avg = sh_idx[sh_idx['Year'] == 1995]['Weighted_Avg'].values
    val_2005_avg = sh_idx[sh_idx['Year'] == 2005]['Weighted_Avg'].values
    if len(val_1995_avg) > 0 and len(val_2005_avg) > 0:
        delta_avg = val_2005_avg[0] - val_1995_avg[0]
        ratio_avg = val_2005_avg[0] / val_1995_avg[0] if val_1995_avg[0] != 0 else np.nan
        print(f"  地域平均: {val_1995_avg[0]:.3f} → {val_2005_avg[0]:.3f}  (Δ{delta_avg:+.3f}, ×{ratio_avg:.2f})")
    else:
        print("  地域平均: データなし")
        print(f"    利用可能なYear: {sorted(sh_idx['Year'].unique().tolist())}")


print_dependency_summary(full_dep, full_idx)


In [ ]:
# =========================================================
# Appendix / robustness:
# 対関東「総依存」と「プロデューサーサービス依存」の比較
# =========================================================

def calculate_kanto_all_vs_ps_dependency():
    records = []
    num_sec = 35
    service_sectors = ['金融・保険', '情報通信', '広告・対事業所サービス']

    for year in sorted(YEAR_FILES.keys()):
        if not os.path.exists(YEAR_FILES[year]):
            continue

        print(f">>> Processing dependency comparison: {year}")
        Z, X, F_list, M, names = get_year_data(year)

        n = len(X)
        X_safe = np.where(X == 0, 1e-9, X)
        A = Z @ np.diag(1 / X_safe)
        Total_D = (A @ X) + np.sum(F_list, axis=0)
        m = np.clip(M / np.where(Total_D <= 0, 1, Total_D), 0, 1)
        L_dom = np.linalg.inv(np.eye(n) - (np.diag(1 - m) @ A))

        # Kanto block indices
        kanto_idx = TARGET_CONFIG['kanto']['idx']
        kanto_slice = slice(kanto_idx * num_sec, (kanto_idx + 1) * num_sec)

        # Kanto producer services indices
        kanto_ps_idx = [np.where(names == s)[0][0] + kanto_idx * num_sec for s in service_sectors]

        for r_key in ALL_REGIONS:
            r_idx = TARGET_CONFIG[r_key]['idx']
            region_slice = slice(r_idx * num_sec, (r_idx + 1) * num_sec)

            X_reg = X[region_slice]
            X_reg_sum = X_reg.sum()
            valid_idx = X_reg > 0

            # sector-wise induced coefficients from Kanto all sectors
            kanto_all_by_sector = L_dom[kanto_slice, region_slice].sum(axis=0)
            kanto_ps_by_sector = L_dom[kanto_ps_idx, region_slice].sum(axis=0)

            # weighted averages
            if X_reg_sum > 0:
                kanto_all_weighted = np.sum(kanto_all_by_sector[valid_idx] * X_reg[valid_idx]) / X_reg_sum * 100
                kanto_ps_weighted = np.sum(kanto_ps_by_sector[valid_idx] * X_reg[valid_idx]) / X_reg_sum * 100
            else:
                kanto_all_weighted = 0
                kanto_ps_weighted = 0

            ps_share = (kanto_ps_weighted / kanto_all_weighted) if kanto_all_weighted > 0 else np.nan

            records.append({
                'Year': int(year),
                'Region': r_key,
                'Kanto_All_Dependency': kanto_all_weighted,
                'Kanto_PS_Dependency': kanto_ps_weighted,
                'PS_Share_in_Kanto_Dependency': ps_share
            })

    return pd.DataFrame(records)

def summarize_dependency_changes(dep_compare_df):
    rows = []
    for r in ALL_REGIONS:
        d = dep_compare_df[dep_compare_df['Region'] == r].sort_values('Year')
        if set([1985, 2005]).issubset(set(d['Year'])):
            d85 = d[d['Year'] == 1985].iloc[0]
            d05 = d[d['Year'] == 2005].iloc[0]
            rows.append({
                'Region': r,
                'All_Dep_1985': d85['Kanto_All_Dependency'],
                'All_Dep_2005': d05['Kanto_All_Dependency'],
                'Delta_All_Dep': d05['Kanto_All_Dependency'] - d85['Kanto_All_Dependency'],
                'PS_Dep_1985': d85['Kanto_PS_Dependency'],
                'PS_Dep_2005': d05['Kanto_PS_Dependency'],
                'Delta_PS_Dep': d05['Kanto_PS_Dependency'] - d85['Kanto_PS_Dependency'],
                'PS_Share_1985': d85['PS_Share_in_Kanto_Dependency'],
                'PS_Share_2005': d05['PS_Share_in_Kanto_Dependency'],
                'Delta_PS_Share': d05['PS_Share_in_Kanto_Dependency'] - d85['PS_Share_in_Kanto_Dependency'],
            })
    return pd.DataFrame(rows)

dep_compare_df = calculate_kanto_all_vs_ps_dependency()

print("\n=== Shikoku: Kanto All vs Producer Services ===")
display(dep_compare_df[dep_compare_df['Region'] == 'shikoku'].sort_values('Year'))

print("\n=== All Regions (1985 and 2005) ===")
display(dep_compare_df[dep_compare_df['Year'].isin([1985, 2005])].sort_values(['Region', 'Year']))

dep_change_summary = summarize_dependency_changes(dep_compare_df)
display(dep_change_summary.sort_values('Delta_PS_Dep', ascending=False))

In [ ]:
# =========================================================
# Sensitivity analysis for Kanto producer-service dependency
# （末尾追記用：既存コードを壊さず追加）
# =========================================================

# ---------------------------------------------------------
# 1. 定義セット
# ---------------------------------------------------------
DEPENDENCY_DEFINITIONS = {
    'baseline_ps': ['金融・保険', '情報通信', '広告・対事業所サービス'],
    'plus_commerce': ['金融・保険', '情報通信', '広告・対事業所サービス', '商業'],
    'plus_commerce_transport': ['金融・保険', '情報通信', '広告・対事業所サービス', '商業', '運輸'],
    'broad_with_realestate': ['金融・保険', '情報通信', '広告・対事業所サービス', '商業', '運輸', '不動産'],
}

DEF_LABELS_EN = {
    'baseline_ps': 'Baseline: Finance + IT + Business services',
    'plus_commerce': '+ Commerce',
    'plus_commerce_transport': '+ Commerce + Transport',
    'broad_with_realestate': '+ Commerce + Transport + Real estate',
}

# ---------------------------------------------------------
# 2. 共通計算関数
# ---------------------------------------------------------
def calculate_dependency_by_definition(definition_sectors, region_key='shikoku'):
    """
    指定した sector set について、
    各年の weighted average dependency を返す。
    """
    results = []
    num_sec = 35

    for year in sorted(YEAR_FILES.keys()):
        if not os.path.exists(YEAR_FILES[year]):
            continue

        Z, X, F_list, M, names = get_year_data(year)

        # レオンチェフ逆行列
        n = len(X)
        X_safe = np.where(X == 0, 1e-9, X)
        A = Z @ np.diag(1 / X_safe)
        Total_D = (A @ X) + np.sum(F_list, axis=0)
        m = np.clip(M / np.where(Total_D <= 0, 1, Total_D), 0, 1)
        L_dom = np.linalg.inv(np.eye(n) - (np.diag(1 - m) @ A))

        # region slice
        r_idx = TARGET_CONFIG[region_key]['idx']
        region_slice = slice(r_idx * num_sec, (r_idx + 1) * num_sec)
        X_reg = X[region_slice]
        X_reg_sum = X_reg.sum()
        valid_idx = X_reg > 0

        # Kanto block
        kanto_idx = TARGET_CONFIG['kanto']['idx']

        # sector indices in Kanto
        kanto_target_idx = []
        for s in definition_sectors:
            hit = np.where(names == s)[0]
            if len(hit) == 0:
                print(f"[WARN] sector not found in {year}: {s}")
                continue
            kanto_target_idx.append(hit[0] + kanto_idx * num_sec)

        if len(kanto_target_idx) == 0:
            weighted_avg = np.nan
        else:
            dep_by_sector = L_dom[kanto_target_idx, region_slice].sum(axis=0)
            weighted_avg = (
                np.sum(dep_by_sector[valid_idx] * X_reg[valid_idx]) / X_reg_sum * 100
                if X_reg_sum > 0 else np.nan
            )

        results.append({
            'Year': int(year),
            'Region': region_key,
            'Definition_Sectors': ', '.join(definition_sectors),
            'Weighted_Avg_Dependency': weighted_avg
        })

    return pd.DataFrame(results)

def calculate_macro_dependency_by_definition(definition_sectors, region_key='shikoku'):
    """
    指定した sector set について、
    マクロ部門別 dependency の年次推移を返す。
    """
    results = []
    num_sec = 35

    ind_to_macro = {
        ind: macro
        for macro, industries in MACRO_GROUPS.items()
        for ind in industries
    }

    for year in sorted(YEAR_FILES.keys()):
        if not os.path.exists(YEAR_FILES[year]):
            continue

        Z, X, F_list, M, names = get_year_data(year)

        # レオンチェフ逆行列
        n = len(X)
        X_safe = np.where(X == 0, 1e-9, X)
        A = Z @ np.diag(1 / X_safe)
        Total_D = (A @ X) + np.sum(F_list, axis=0)
        m = np.clip(M / np.where(Total_D <= 0, 1, Total_D), 0, 1)
        L_dom = np.linalg.inv(np.eye(n) - (np.diag(1 - m) @ A))

        r_idx = TARGET_CONFIG[region_key]['idx']
        region_slice = slice(r_idx * num_sec, (r_idx + 1) * num_sec)

        # Kanto側 sector index
        kanto_idx = TARGET_CONFIG['kanto']['idx']
        kanto_target_idx = []
        for s in definition_sectors:
            hit = np.where(names == s)[0]
            if len(hit) == 0:
                continue
            kanto_target_idx.append(hit[0] + kanto_idx * num_sec)

        if len(kanto_target_idx) == 0:
            continue

        dep_by_sector = L_dom[kanto_target_idx, region_slice].sum(axis=0) * 100

        tmp = pd.DataFrame({
            'Industry': names,
            'Dependency': dep_by_sector
        })
        tmp['Macro_Group'] = tmp['Industry'].map(ind_to_macro)

        macro_mean = tmp.groupby('Macro_Group', dropna=False)['Dependency'].mean().reset_index()
        for _, row in macro_mean.iterrows():
            results.append({
                'Year': int(year),
                'Region': region_key,
                'Macro_Group': row['Macro_Group'],
                'Weighted_Avg_Dependency': row['Dependency'],
                'Definition_Sectors': ', '.join(definition_sectors)
            })

    return pd.DataFrame(results)

# ---------------------------------------------------------
# 3. 全定義をまとめて計算
# ---------------------------------------------------------
def run_dependency_sensitivity_analysis(region_key='shikoku'):
    summary_list = []
    macro_list = []

    for def_key, sectors in DEPENDENCY_DEFINITIONS.items():
        print(f">>> Running sensitivity: {def_key} / {sectors}")

        df_summary = calculate_dependency_by_definition(sectors, region_key=region_key)
        df_summary['Definition_Key'] = def_key
        df_summary['Definition_Label_EN'] = DEF_LABELS_EN.get(def_key, def_key)
        summary_list.append(df_summary)

        df_macro = calculate_macro_dependency_by_definition(sectors, region_key=region_key)
        df_macro['Definition_Key'] = def_key
        df_macro['Definition_Label_EN'] = DEF_LABELS_EN.get(def_key, def_key)
        macro_list.append(df_macro)

    dep_sensitivity_summary = pd.concat(summary_list, ignore_index=True)
    dep_sensitivity_macro = pd.concat(macro_list, ignore_index=True)

    return dep_sensitivity_summary, dep_sensitivity_macro

dep_sensitivity_summary, dep_sensitivity_macro = run_dependency_sensitivity_analysis(region_key='shikoku')

# ---------------------------------------------------------
# 4. CSV保存
# ---------------------------------------------------------
tables_dir = os.path.join(SAVE_DIR, 'tables')
os.makedirs(tables_dir, exist_ok=True)

dep_sensitivity_summary.to_csv(
    os.path.join(tables_dir, 'dependency_sensitivity_summary_shikoku.csv'),
    index=False, encoding='utf-8-sig'
)

dep_sensitivity_macro.to_csv(
    os.path.join(tables_dir, 'dependency_sensitivity_macro_shikoku.csv'),
    index=False, encoding='utf-8-sig'
)

print("\n=== Sensitivity summary (Shikoku) ===")
display(dep_sensitivity_summary)

# ---------------------------------------------------------
# 5. 1985→2005 の比較表を作成
# ---------------------------------------------------------
def make_sensitivity_change_table(dep_sensitivity_summary):
    rows = []
    for def_key in dep_sensitivity_summary['Definition_Key'].unique():
        d = dep_sensitivity_summary[dep_sensitivity_summary['Definition_Key'] == def_key].sort_values('Year')
        d85 = d[d['Year'] == 1985]
        d05 = d[d['Year'] == 2005]
        if len(d85) == 0 or len(d05) == 0:
            continue

        v85 = d85.iloc[0]['Weighted_Avg_Dependency']
        v05 = d05.iloc[0]['Weighted_Avg_Dependency']

        rows.append({
            'Definition_Key': def_key,
            'Definition_Label_EN': d05.iloc[0]['Definition_Label_EN'],
            '1985': v85,
            '2005': v05,
            'Delta_1985_2005': v05 - v85,
            'Ratio_2005_1985': v05 / v85 if pd.notna(v85) and v85 != 0 else np.nan
        })
    return pd.DataFrame(rows)

dep_sensitivity_change = make_sensitivity_change_table(dep_sensitivity_summary)
dep_sensitivity_change.to_csv(
    os.path.join(tables_dir, 'dependency_sensitivity_change_shikoku.csv'),
    index=False, encoding='utf-8-sig'
)

print("\n=== Sensitivity change table (1985 -> 2005) ===")
display(dep_sensitivity_change)

# ---------------------------------------------------------
# 6. 折れ線グラフ
# ---------------------------------------------------------
def plot_dependency_sensitivity_summary(dep_sensitivity_summary, region_key='shikoku', lang='en'):
    save_dir = os.path.join(SAVE_DIR, 'dependency_analysis')
    os.makedirs(save_dir, exist_ok=True)

    plt.figure(figsize=(11, 7))

    line_styles = ['-', '--', '-.', ':']
    markers = ['o', 's', '^', 'D']

    for i, def_key in enumerate(dep_sensitivity_summary['Definition_Key'].unique()):
        d = dep_sensitivity_summary[dep_sensitivity_summary['Definition_Key'] == def_key].sort_values('Year')
        plt.plot(
            d['Year'],
            d['Weighted_Avg_Dependency'],
            label=d['Definition_Label_EN'].iloc[0],
            linewidth=2.8,
            linestyle=line_styles[i % len(line_styles)],
            marker=markers[i % len(markers)],
            markersize=8,
            color='black' if i == 0 else None
        )

    plt.ylabel("Dependency on Kanto service sectors (coefficient ×100)", fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=11)

    plt.tight_layout()
    plt.savefig(
        os.path.join(save_dir, f'shikoku_dependency_sensitivity_summary_{lang}.png'),
        dpi=300, bbox_inches='tight'
    )
    plt.close()

plot_dependency_sensitivity_summary(dep_sensitivity_summary)

# ---------------------------------------------------------
# 7. マクロ部門別比較グラフ（2005年）
# ---------------------------------------------------------
def plot_dependency_sensitivity_macro_2005(dep_sensitivity_macro, lang='en'):
    save_dir = os.path.join(SAVE_DIR, 'dependency_analysis')
    os.makedirs(save_dir, exist_ok=True)

    df = dep_sensitivity_macro[dep_sensitivity_macro['Year'] == 2005].copy()
    if df.empty:
        return

    macro_order = list(MACRO_GROUPS.keys())
    macro_labels = [MACRO_EN.get(m, m) for m in macro_order]

    defs = list(df['Definition_Key'].unique())
    x = np.arange(len(macro_order))
    width = 0.18

    fig, ax = plt.subplots(figsize=(12, 7))

    for i, def_key in enumerate(defs):
        sub = df[df['Definition_Key'] == def_key].copy()
        sub = sub.set_index('Macro_Group').reindex(macro_order)

        ax.bar(
            x + i * width - width * (len(defs)-1) / 2,
            sub['Weighted_Avg_Dependency'].values,
            width=width,
            label=sub['Definition_Label_EN'].dropna().iloc[0] if len(sub) > 0 else def_key,
            edgecolor='black'
        )

    ax.set_xticks(x)
    ax.set_xticklabels(macro_labels, rotation=15, fontsize=12)
    ax.set_ylabel("Dependency on Kanto service sectors (coefficient ×100)", fontsize=14)
    ax.grid(axis='y', linestyle=':', alpha=0.4)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)

    plt.tight_layout()
    plt.savefig(
        os.path.join(save_dir, f'shikoku_dependency_sensitivity_macro_2005_{lang}.png'),
        dpi=300, bbox_inches='tight'
    )
    plt.close()

plot_dependency_sensitivity_macro_2005(dep_sensitivity_macro)

# ---------------------------------------------------------
# 8. 論文用の簡易print
# ---------------------------------------------------------
print("\n" + "="*70)
print("【Sensitivity analysis summary for manuscript】")
print("="*70)

for _, row in dep_sensitivity_change.sort_values('Definition_Key').iterrows():
    print(
        f"{row['Definition_Label_EN']}: "
        f"1985={row['1985']:.3f}, 2005={row['2005']:.3f}, "
        f"Δ={row['Delta_1985_2005']:+.3f}, ×{row['Ratio_2005_1985']:.2f}"
    )

## 架橋分析

In [ ]:
# =========================================================
# 四国架橋分析セクション（修正版：詳細列名対応）
# =========================================================

# --- 1. テキスト・配色設定 ---
def get_bridge_texts(lang, year1, year2):
    if lang == 'ja':
        return {
            "legend_ss": "Rate Effect（自給率変化）",
            "legend_share": "シェア効果（需要構造変化）",
            "ylab_ssr_change": "ΔSSRへの寄与（pp）",
            "scatter_xlabel": "構造変化要因 (Share Effect)\n(between-sector composition)\n← 産業シェアの縮小 | 産業シェアの拡大 →",
            "scatter_ylabel": "競争力要因 (Rate Effect)\n(within-sector SSR change)\n← 自給率の低下 | 自給率の向上 →",
            "q1": "【成長牽引】\nシェア拡大・自給率向上", "q2": "【構造的先鋭化】\nシェア縮小・自給率向上",
            "q3": "【二重の衰退】\nシェア縮小・自給率低下", "q4": "【空洞的成長】\nシェア拡大・自給率低下"
        }
    else:
        return {
            "legend_ss": "Rate Effect",
            "legend_share": "Share Effect",
            "ylab_ssr_change": "Contribution to ΔSSR (pp)",
            "scatter_xlabel": "Share Effect (Between-Sector Composition)\n← Shrinking Share | Expanding Share →",
            "scatter_ylabel": "Rate Effect (Within-Sector SSR Change)\n← Declining SSR | Rising SSR →",
            "q1": "[Growth Driver]\nExpanding + Rising", "q2": "[Structural Sharpening]\nShrinking + Rising",
            "q3": "[Double Decline]\nShrinking + Declining", "q4": "[Hollow Growth]\nExpanding + Declining"
        }

def get_bridge_colors(style='bw'):
    if style == 'color':
        return {'pos': '#66b3ff', 'neg': '#ff9999', 'ss': 'salmon', 'share': 'skyblue',
                'scatter': {'q1': '#2ecc71', 'q2': '#3498db', 'q3': '#e74c3c', 'q4': '#f39c12'},
                'edge': 'white', 'alpha': 0.75}, {'ss': '', 'share': ''}
    else:
        return {'pos': '#444444', 'neg': '#AAAAAA', 'ss': 'white', 'share': '#E0E0E0',
                'scatter': {'q1': '#000000', 'q2': '#444444', 'q3': '#AAAAAA', 'q4': '#DDDDDD'},
                'edge': 'black', 'alpha': 0.6}, {'ss': '///', 'share': '..'}

# --- 2. 分析・可視化クラス ---
class BridgeAnalyzer:
    def __init__(self, region='shikoku', year1=1985, year2=1990, lang='en', style='bw'):
        self.region = region
        self.year1 = year1
        self.year2 = year2
        self.lang = lang
        self.style = style

        self.t = get_bridge_texts(lang, year1, year2)
        self.colors, self.hatches = get_bridge_colors(style)

        self.save_dir = os.path.join(SAVE_DIR, 'bridge_analysis', 'figures_for_paper')
        os.makedirs(self.save_dir, exist_ok=True)

        # クラス共通の列名定義（長い名称を使用）
        self.col_share = 'Share Effect (between-sector composition)'
        self.col_rate = 'Rate Effect (within-sector SSR change)'

    def _get_sector_display_name(self, sector):
        return SECTOR_EN.get(sector, sector) if self.lang == 'en' else sector

    def verify_interaction_magnitude(self):
        res1 = get_structure(self.year1, self.region)['skyline']
        res2 = get_structure(self.year2, self.region)['skyline']

        comp = pd.merge(res1, res2, on='sector', suffixes=('_1', '_2'), how='outer').fillna(0)
        comp['delta_share'] = comp['share_2'] - comp['share_1']
        comp['delta_ss'] = comp['ss_2'] - comp['ss_1']
        comp['term_a'] = comp['delta_share'] * comp['ss_1']
        comp['term_b'] = comp['share_1'] * comp['delta_ss']
        comp['interaction'] = comp['delta_share'] * comp['delta_ss']

        total_change = (comp['share_2'] * comp['ss_2']).sum() - (comp['share_1'] * comp['ss_1']).sum()
        total_abs_impact = comp['term_a'].abs().sum() + comp['term_b'].abs().sum() + comp['interaction'].abs().sum()
        abs_interaction_ratio = comp['interaction'].abs().sum() / total_abs_impact if total_abs_impact != 0 else 0

        print(f"--- Detailed Check: {self.region} ({self.year1}-{self.year2}) ---")
        print(f"Interaction Ratio (Absolute Impact): {abs_interaction_ratio:.2%}")
        return abs_interaction_ratio

    def prepare_comparison_data(self):
        res1 = get_structure(self.year1, self.region)['skyline']
        res2 = get_structure(self.year2, self.region)['skyline']

        comp = pd.merge(
            res1[['sector', 'share', 'ss']],
            res2[['sector', 'share', 'ss']],
            on='sector', how='outer',
            suffixes=(f'_{self.year1}', f'_{self.year2}')
        ).fillna(0)

        comp[self.col_share] = (
            (comp[f'share_{self.year2}'] - comp[f'share_{self.year1}']) *
            ((comp[f'ss_{self.year1}'] + comp[f'ss_{self.year2}']) / 2)
        )
        comp[self.col_rate] = (
            (comp[f'ss_{self.year2}'] - comp[f'ss_{self.year1}']) *
            ((comp[f'share_{self.year1}'] + comp[f'share_{self.year2}']) / 2)
        )
        comp['Contribution'] = comp[self.col_share] + comp[self.col_rate]
        comp['sector_display'] = comp['sector'].apply(self._get_sector_display_name)
        return comp

    def prepare_regional_decomposition(self, compare_regions):
        all_regions = [self.region] + compare_regions
        decomp_list = []

        for reg in all_regions:
            res1 = get_structure(self.year1, reg)['skyline'].set_index('sector')
            res2 = get_structure(self.year2, reg)['skyline'].set_index('sector')

            share_effect = ((res2['share'] - res1['share']) * (res1['ss'] + res2['ss']) / 2).sum()
            ss_effect = ((res2['ss'] - res1['ss']) * (res1['share'] + res2['share']) / 2).sum()
            total_change = (res2['share'] * res2['ss']).sum() - (res1['share'] * res1['ss']).sum()

            decomp_list.append({
                'Region': reg,
                'Region_Display': TRANSLATIONS[self.lang]['regions'][reg],
                'SSR_Change': total_change,
                'Share_Effect': share_effect,
                'Rate_Effect': ss_effect
            })
        return pd.DataFrame(decomp_list)

    def plot_combined_regional_comparison(self, decomp_short, decomp_long):
            fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
            all_vals = np.concatenate([decomp_short['Share_Effect'].values * 100, decomp_short['Rate_Effect'].values * 100,
                                      decomp_long['Share_Effect'].values * 100, decomp_long['Rate_Effect'].values * 100])
            margin = (all_vals.max() - all_vals.min()) * 0.15
            xlim = (all_vals.min() - margin, all_vals.max() + margin)

            for i, (ax, df, title) in enumerate(zip(axes, [decomp_short, decomp_long], [f'1985–1990 (Short-term)', f'1985–2005 (Long-term)'])):
                share_pct, ss_pct = df['Share_Effect'].values * 100, df['Rate_Effect'].values * 100
                y_pos, zeros = np.arange(len(df)), np.zeros(len(df))

                ss_pos, share_pos = np.maximum(ss_pct, zeros), np.maximum(share_pct, zeros)
                ax.barh(y_pos, ss_pos, color=self.colors['ss'], hatch=self.hatches['ss'], edgecolor='black', label=self.t['legend_ss'] if i == 1 else "")
                ax.barh(y_pos, share_pos, left=ss_pos, color=self.colors['share'], hatch=self.hatches['share'], edgecolor='black', label=self.t['legend_share'] if i == 1 else "")

                ss_neg, share_neg = np.minimum(ss_pct, zeros), np.minimum(share_pct, zeros)
                ax.barh(y_pos, ss_neg, color=self.colors['ss'], hatch=self.hatches['ss'], edgecolor='black')
                ax.barh(y_pos, share_neg, left=ss_neg, color=self.colors['share'], hatch=self.hatches['share'], edgecolor='black')

                if i == 0:
                    ax.set_yticks(y_pos)
                    ax.set_yticklabels(df['Region_Display'], fontsize=16)

                ax.set_xlabel(self.t['ylab_ssr_change'], fontsize=16)
                ax.set_title(title, fontsize=16, fontweight='bold')
                ax.set_xlim(xlim)
                ax.axvline(0, color='black', linewidth=0.8)
                ax.grid(axis='x', ls=':', alpha=0.3)

                ax.tick_params(axis='x', labelsize=14)

                if i == 1:
                    ax.legend(loc='best', frameon=True, fontsize=14)

            plt.tight_layout()
            save_path = os.path.join(self.save_dir, f'regional_comparison_combined_{self.lang}_{self.style}.png')
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            plt.close()

    def plot_fig2_temporal_comparison(self, short_comp, long_comp):
        fig, ax = plt.subplots(figsize=(10, 6))
        ss_short, share_short = short_comp[self.col_rate].sum() * 100, short_comp[self.col_share].sum() * 100
        ss_long, share_long = long_comp[self.col_rate].sum() * 100, long_comp[self.col_share].sum() * 100
        width = 0.6

        def plot_bars(idx, ss_val, share_val):
            ax.bar(idx, max(0, ss_val), width, color=self.colors['ss'], hatch=self.hatches['ss'], edgecolor='black', label=self.t['legend_ss'] if idx==0 else "")
            ax.bar(idx, max(0, share_val), width, bottom=max(0, ss_val), color=self.colors['share'], hatch=self.hatches['share'], edgecolor='black', label=self.t['legend_share'] if idx==0 else "")
            ax.bar(idx, min(0, ss_val), width, color=self.colors['ss'], hatch=self.hatches['ss'], edgecolor='black')
            ax.bar(idx, min(0, share_val), width, bottom=min(0, ss_val), color=self.colors['share'], hatch=self.hatches['share'], edgecolor='black')

        plot_bars(0, ss_short, share_short)
        plot_bars(1, ss_long, share_long)

        ax.set_xticks([0, 1])
        ax.set_xticklabels(['1985–1990', '1985–2005'], fontsize=14)
        ax.set_ylabel(self.t['ylab_ssr_change'], fontsize=14)
        ax.axhline(0, color='black', linewidth=0.8)
        ax.legend(loc='best', frameon=True)
        ax.grid(axis='y', ls=':', alpha=0.3)
        plt.tight_layout()
        plt.savefig(
            os.path.join(self.save_dir, f'shikoku_temporal_decomposition_{self.lang}_{self.style}.png'),
            dpi=300,
            bbox_inches='tight'
        )
        plt.close()

    def plot_fig3_or_fig4_sectoral(self, comp, effect_type='negative', n=5):
        target_col = self.col_rate

        if effect_type == 'negative':
            data = comp.nsmallest(n, target_col).sort_values(target_col)
            title_suffix = 'Negative Rate Contributors'
            file_stub = 'sectoral_negative_rate_contributors'
        else:
            data = comp.nlargest(n, target_col).sort_values(target_col, ascending=False)
            title_suffix = 'Positive Rate Contributors'
            file_stub = 'sectoral_positive_rate_contributors'

        fig, ax = plt.subplots(figsize=(10, 6))
        colors = [self.colors['neg'] if x < 0 else self.colors['pos'] for x in data[target_col]]
        ax.barh(data['sector_display'], data[target_col] * 100, color=colors, edgecolor='black', linewidth=0.8)
        ax.set_xlabel('Rate Effect Contribution (%)', fontsize=12)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.grid(axis='x', ls=':', alpha=0.5)
        plt.tight_layout()
        plt.savefig(
            os.path.join(
                self.save_dir,
                f'{file_stub}_{self.year1}_{self.year2}_{self.lang}_{self.style}.png'
            ),
            dpi=300,
            bbox_inches='tight'
        )
        plt.close()

    def plot_scatter_for_section45(self, comp, threshold=0.002):
        fig, ax = plt.subplots(figsize=(10, 8))
        q_colors = self.colors['scatter']
        texts = []

        def plus_formatter(x, pos):
            if x > 0:
                return f'+{x:g}'
            return f'{x:g}'

        ax.xaxis.set_major_formatter(FuncFormatter(plus_formatter))
        ax.yaxis.set_major_formatter(FuncFormatter(plus_formatter))

        for _, row in comp.iterrows():
            x, y = row[self.col_share], row[self.col_rate]

            if x > 0 and y > 0:
                color = q_colors['q1']
            elif x < 0 and y > 0:
                color = q_colors['q2']
            elif x < 0 and y < 0:
                color = q_colors['q3']
            else:
                color = q_colors['q4']

            ax.scatter(x * 100, y * 100, s=200, color=color, edgecolor=self.colors['edge'], linewidth=1.5, alpha=self.colors['alpha'], zorder=3)
            if abs(x) > threshold or abs(y) > threshold:
                texts.append(ax.text(x * 100, y * 100, row['sector_display'], fontsize=10, ha='center', va='center'))

        adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))
        ax.axhline(0, color='black', linewidth=1.2, zorder=2)
        ax.axvline(0, color='black', linewidth=1.2, zorder=2)

        xlim, ylim = ax.get_xlim(), ax.get_ylim()
        quadrant_info = [
            (xlim[1]*0.7, ylim[1]*0.85, self.t['q1'], q_colors['q1']),
            (xlim[0]*0.7, ylim[1]*0.85, self.t['q2'], q_colors['q2']),
            (xlim[0]*0.7, ylim[0]*0.85, self.t['q3'], q_colors['q3']),
            (xlim[1]*0.7, ylim[0]*0.85, self.t['q4'], q_colors['q4'])
        ]

        for x_pos, y_pos, label, bg_color in quadrant_info:
            ax.text(x_pos, y_pos, label, fontsize=12, ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.5', facecolor=bg_color, edgecolor='gray', alpha=0.3))

        ax.set_xlabel(self.t['scatter_xlabel'], fontsize=16)
        ax.set_ylabel(self.t['scatter_ylabel'], fontsize=16)
        ax.grid(True, ls=':', alpha=0.3)
        plt.tight_layout()
        plt.savefig(
            os.path.join(
                self.save_dir,
                f'structural_transformation_scatter_{self.year1}_{self.year2}_{self.lang}_{self.style}.png'
            ),
            dpi=300,
            bbox_inches='tight'
        )
        plt.close()

# 実質価格化処理
    def prepare_real_comparison_data(self):
        mgr = GLOBAL_DEFLATOR_MANAGER
        if mgr.deflators is None:
            raise ValueError("DeflatorManagerが未初期化です。")

        y0, y1 = str(self.year1), str(self.year2)

        sky1 = get_structure(y0, self.region)['skyline'].set_index('sector')
        sky2 = get_structure(y1, self.region)['skyline'].set_index('sector')

        sectors = sky1.index
        defs_t0 = mgr.deflators.reindex(sectors)[y0].fillna(1.0)
        defs_t1 = mgr.deflators.reindex(sectors)[y1].fillna(1.0)

        real_share_eff, real_rate_eff = calculate_real_factor_decomposition(
            X_nom_t0=sky1['x'],
            X_nom_t1=sky2['x'],
            XD_nom_t0=sky1['xd'],
            XD_nom_t1=sky2['xd'],
            deflators_t0=defs_t0,
            deflators_t1=defs_t1,
        )

        nom_share_eff = (sky2['share'] - sky1['share']) * (sky1['ss'] + sky2['ss']) / 2
        nom_rate_eff  = (sky2['ss']    - sky1['ss'])    * (sky1['share'] + sky2['share']) / 2

        comp = pd.DataFrame({
            'sector': sectors,
            'sector_display': [self._get_sector_display_name(s) for s in sectors],
            'nom_share_effect': nom_share_eff.values,
            'nom_rate_effect': nom_rate_eff.values,
            'real_share_effect': real_share_eff,
            'real_rate_effect': real_rate_eff,
        })
        comp['nom_contribution'] = comp['nom_share_effect'] + comp['nom_rate_effect']
        comp['real_contribution'] = comp['real_share_effect'] + comp['real_rate_effect']
        return comp

    def prepare_real_regional_decomposition(self, compare_regions):
        mgr = GLOBAL_DEFLATOR_MANAGER
        y0, y1 = str(self.year1), str(self.year2)
        all_regions = [self.region] + compare_regions
        decomp_list = []

        for reg in all_regions:
            sky1 = get_structure(y0, reg)['skyline'].set_index('sector')
            sky2 = get_structure(y1, reg)['skyline'].set_index('sector')

            sectors = sky1.index
            defs_t0 = mgr.deflators.reindex(sectors)[y0].fillna(1.0)
            defs_t1 = mgr.deflators.reindex(sectors)[y1].fillna(1.0)

            real_share_eff, real_rate_eff = calculate_real_factor_decomposition(
                X_nom_t0=sky1['x'],
                X_nom_t1=sky2['x'],
                XD_nom_t0=sky1['xd'],
                XD_nom_t1=sky2['xd'],
                deflators_t0=defs_t0,
                deflators_t1=defs_t1,
            )

            nom_share_eff = ((sky2['share'] - sky1['share']) * (sky1['ss'] + sky2['ss']) / 2).sum()
            nom_rate_eff  = ((sky2['ss'] - sky1['ss']) * (sky1['share'] + sky2['share']) / 2).sum()
            total_change  = (sky2['share'] * sky2['ss']).sum() - (sky1['share'] * sky1['ss']).sum()

            decomp_list.append({
                'Region': reg,
                'Region_Display': TRANSLATIONS[self.lang]['regions'][reg],
                'Total_Change': total_change,
                'Nominal_Share': nom_share_eff,
                'Nominal_Rate': nom_rate_eff,
                'Real_Share': real_share_eff.sum(),
                'Real_Rate': real_rate_eff.sum()
            })

        return pd.DataFrame(decomp_list)

    def plot_appendix_robustness(self, comp_real):
        """Appendix Figure: 名目vs実質の要因分解比較（集計値）"""
        nom_share  = comp_real['nom_share_effect'].sum()  * 100
        nom_rate   = comp_real['nom_rate_effect'].sum()   * 100
        real_share = comp_real['real_share_effect'].sum() * 100
        real_rate  = comp_real['real_rate_effect'].sum()  * 100

        labels = ['Nominal', 'Real (deflated wi)']
        share_vals = [nom_share,  real_share]
        rate_vals  = [nom_rate,   real_rate]

        x = np.arange(len(labels))
        width = 0.35

        fig, ax = plt.subplots(figsize=(8, 6))
        b1 = ax.bar(x - width/2, share_vals, width,
                    label='Share Effect', **({'color': self.colors['share']} if self.style == 'color'
                                            else {'color': self.colors['share'], 'hatch': self.hatches['share']}),
                    edgecolor='black')
        b2 = ax.bar(x + width/2, rate_vals, width,
                    label='Rate Effect',  **({'color': self.colors['ss']} if self.style == 'color'
                                            else {'color': self.colors['ss'], 'hatch': self.hatches['ss']}),
                    edgecolor='black')

        for bar in list(b1) + list(b2):
            v = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2,
                    v + (0.3 if v >= 0 else -0.6),
                    f'{v:+.2f} pp', ha='center', fontsize=11)

        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=13)
        ax.set_ylabel('Contribution to ∆SSR (pp)', fontsize=12)
        # ax.set_title(
        #     f'Appendix: Robustness Check — Nominal vs. Real-weight Decomposition\n'
        #     f'Shikoku {self.year1}–{self.year2}',
        #     fontsize=13, fontweight='bold'
        # )
        ax.legend(fontsize=11)
        ax.grid(axis='y', linestyle='--', alpha=0.3)
        plt.tight_layout()
        plt.savefig(
            os.path.join(self.save_dir, f'appendix_robustness_{self.year1}{self.year2}_{self.lang}_{self.style}.png'),
            dpi=300, bbox_inches='tight'
        )
        plt.close()
        print(f"  Share Effect: Nominal={nom_share:+.3f} pp → Real={real_share:+.3f} pp")
        print(f"  Rate Effect:  Nominal={nom_rate:+.3f} pp → Real={real_rate:+.3f} pp")

# --- 3. 実行パイプライン（全地域検証対応版） ---
def run_bridge_analysis_pipeline():
    primary_region = 'shikoku'
    compare_regions = ['chugoku', 'tohoku', 'kyushu']
    all_regions = [primary_region] + compare_regions

    # --- 全地域の交互作用を一括検証 ---
    print("="*60)
    print("CHECKING INTERACTION MAGNITUDE FOR ALL REGIONS")
    print("="*60)
    for period_name, y1, y2 in [("Short-term (1985-1990)", 1985, 1990), ("Long-term (1985-2005)", 1985, 2005)]:
        print(f"\n[Period: {period_name}]")
        for reg in all_regions:
            # 各地域用に一時的にアナライザーを生成して検証
            v_analyzer = BridgeAnalyzer(region=reg, year1=y1, year2=y2)
            v_analyzer.verify_interaction_magnitude()
    print("="*60 + "\n")

    # 以降、可視化ループ
    for lang in RUN_CONFIG['languages']['bridge']:
        for style in RUN_CONFIG['styles']['bridge']:
            print(f">>> グラフ生成開始: 言語={lang}, スタイル={style}")

            # 短期分析 (1985-1990)
            analyzer_short = BridgeAnalyzer(region=primary_region, year1=1985, year2=1990, lang=lang, style=style)
            comp_short = analyzer_short.prepare_comparison_data()
            analyzer_short.plot_fig3_or_fig4_sectoral(comp_short, effect_type='negative', n=5)
            decomp_short = analyzer_short.prepare_regional_decomposition(compare_regions)

            # 長期分析 (1985-2005)
            analyzer_long = BridgeAnalyzer(region=primary_region, year1=1985, year2=2005, lang=lang, style=style)
            comp_long = analyzer_long.prepare_comparison_data()
            analyzer_long.plot_fig3_or_fig4_sectoral(comp_long, effect_type='positive', n=5)
            decomp_long = analyzer_long.prepare_regional_decomposition(compare_regions)

            # 統合グラフ描画
            analyzer_long.plot_combined_regional_comparison(decomp_short, decomp_long)
            analyzer_long.plot_fig2_temporal_comparison(comp_short, comp_long)
            analyzer_long.plot_scatter_for_section45(comp_long, threshold=0.002)

            # CSV出力 (最終ループで保存)
            if lang == 'en' and style == 'color':
                tables_dir = os.path.join(SAVE_DIR, 'bridge_analysis', 'tables')
                os.makedirs(tables_dir, exist_ok=True)
                comp_short.to_csv(f"{tables_dir}/bridge_analysis_1985_1990.csv", index=False, encoding='utf-8-sig')
                comp_long.to_csv(f"{tables_dir}/bridge_analysis_1985_2005.csv", index=False, encoding='utf-8-sig')
                decomp_long.to_csv(f"{tables_dir}/bridge_regional_1985_2005.csv", index=False, encoding='utf-8-sig')

            # langループの内側（analyzer_long の処理の後）
            # Appendix用頑健性チェック（論文用なのでenかつcolorの代表で1回実行）
            if lang == 'en' and style == 'color':
                print("  >>> Generating Appendix Robustness Check...")
                comp_real = analyzer_long.prepare_real_comparison_data()
                analyzer_long.plot_appendix_robustness(comp_real)

                # 全地域の実質値・名目値比較表を出力
                real_regional_comp = analyzer_long.prepare_real_regional_decomposition(compare_regions)
                print("\n  [Appendix: Regional Comparison (Nominal vs Real)]")
                display(real_regional_comp)

                tables_dir = os.path.join(SAVE_DIR, 'bridge_analysis', 'tables')
                os.makedirs(tables_dir, exist_ok=True)
                comp_real.to_csv(f"{tables_dir}/appendix_robustness_1985_2005.csv", index=False, encoding='utf-8-sig')
                real_regional_comp.to_csv(f"{tables_dir}/appendix_real_regional_comparison_1985_2005.csv", index=False, encoding='utf-8-sig')

run_bridge_analysis_pipeline()
print("四国架橋分析が完了しました。")

In [ ]:
# =========================================================
# Sharpening Index の地域間比較セクション
# =========================================================

def calculate_regional_sharpening_indices(regions=None, year0='1985', year1='2005'):
    """
    各地域について Sharpening Index を計算し、1つの縦長DataFrameで返す。
    """
    if regions is None:
        regions = ['shikoku', 'tohoku', 'kyushu', 'chugoku']

    all_rows = []

    for region in regions:
        print(f">>> Calculating Sharpening Index: {region} ({year0}->{year1})")

        sky0 = get_structure(year0, region)['skyline']
        sky1 = get_structure(year1, region)['skyline']

        phi_df = calc_sharpening_index(sky0, sky1).copy()
        phi_df['Region'] = region
        phi_df['Region_Display'] = TRANSLATIONS['en']['regions'].get(region, region)

        # 四象限分類
        def classify(row):
            if row['d_share'] < 0 and row['d_ss'] > 0:
                return 'Q2 Structural Sharpening'
            elif row['d_share'] < 0 and row['d_ss'] < 0:
                return 'Q3 Double Decline'
            elif row['d_share'] > 0 and row['d_ss'] > 0:
                return 'Q1 Growth Driver'
            elif row['d_share'] > 0 and row['d_ss'] < 0:
                return 'Q4 Hollow Growth'
            else:
                return 'Boundary/Zero'

        phi_df['quadrant'] = phi_df.apply(classify, axis=1)

        # sharpening の質的分類
        def classify_phi(row):
            phi = row['phi']
            if pd.isna(phi):
                return 'NA'
            if row['d_share'] < 0 and row['d_ss'] > 0:
                if phi > 1:
                    return 'Active sharpening'
                elif phi > 0:
                    return 'Defensive sharpening'
                else:
                    return 'Other'
            elif row['d_share'] < 0 and row['d_ss'] < 0:
                return 'Double decline'
            else:
                return 'Other'

        phi_df['sharpening_type'] = phi_df.apply(classify_phi, axis=1)

        all_rows.append(phi_df)

    result = pd.concat(all_rows, ignore_index=True)
    return result

def summarize_regional_sharpening(phi_all_df):
    """
    本文またはAppendix用の地域別要約表を作る。
    """
    summary_rows = []

    for region in phi_all_df['Region'].unique():
        df = phi_all_df[phi_all_df['Region'] == region].copy()

        q2 = df[df['quadrant'] == 'Q2 Structural Sharpening']
        active = q2[q2['phi'] > 1]
        defensive = q2[(q2['phi'] > 0) & (q2['phi'] <= 1)]

        summary_rows.append({
            'Region': region,
            'Region_Display': TRANSLATIONS['en']['regions'].get(region, region),
            'Num_Q2_Sectors': len(q2),
            'Num_Active_Sharpening': len(active),
            'Num_Defensive_Sharpening': len(defensive),
            'Mean_Phi_Q2': q2['phi'].mean(),
            'Median_Phi_Q2': q2['phi'].median(),
            'Top_Q2_Sector_1': q2.sort_values('phi', ascending=False)['sector'].iloc[0] if len(q2) >= 1 else None,
            'Top_Q2_Sector_2': q2.sort_values('phi', ascending=False)['sector'].iloc[1] if len(q2) >= 2 else None,
        })

    return pd.DataFrame(summary_rows)

def export_sharpening_tables(phi_all_df, summary_df):
    out_dir = os.path.join(SAVE_DIR, 'tables')
    os.makedirs(out_dir, exist_ok=True)

    phi_all_df.to_csv(
        os.path.join(out_dir, 'appendix_sharpening_index_regional_detail_1985_2005.csv'),
        index=False, encoding='utf-8-sig'
    )

    summary_df.to_csv(
        os.path.join(out_dir, 'appendix_sharpening_index_regional_summary_1985_2005.csv'),
        index=False, encoding='utf-8-sig'
    )

    print("\n>>> Sharpening comparison tables exported.")
    display(summary_df)

def plot_regional_sharpening_summary(summary_df, lang='en', style='color'):
    save_dir = os.path.join(SAVE_DIR, 'interregional_comparison')
    os.makedirs(save_dir, exist_ok=True)

    df = summary_df.copy()
    labels = df['Region_Display'].values
    x = np.arange(len(df))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))

    if style == 'color':
        c1, c2 = '#87CEEB', '#FF9999'
    else:
        c1, c2 = 'white', '#BBBBBB'

    ax.bar(x - width/2, df['Num_Q2_Sectors'], width,
           label='Q2 sectors',
           color=c1, edgecolor='black')
    ax.bar(x + width/2, df['Num_Active_Sharpening'], width,
           label='Active sharpening',
           color=c2, edgecolor='black')

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=12)
    ax.set_ylabel('Number of sectors', fontsize=13)
    ax.legend()
    ax.grid(axis='y', linestyle=':', alpha=0.4)

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'compare_sharpening_summary_{lang}_{style}.png'),
                dpi=300, bbox_inches='tight')
    plt.close()

    print(">>> Saved sharpening summary plot.")

# =========================================================
# 実行ブロック：Sharpening Index の地域間比較
# =========================================================
regions_for_comparison = ['shikoku', 'tohoku', 'kyushu', 'chugoku']

phi_all_df = calculate_regional_sharpening_indices(
    regions=regions_for_comparison,
    year0='1985',
    year1='2005'
)

sharpening_summary_df = summarize_regional_sharpening(phi_all_df)

export_sharpening_tables(phi_all_df, sharpening_summary_df)
for lang in RUN_CONFIG['languages']['sharpening_summary']:
    for style in RUN_CONFIG['styles']['sharpening_summary']:
        plot_regional_sharpening_summary(sharpening_summary_df, lang=lang, style=style)

def prepare_sharpening_summary_for_latex(summary_df):
    df = summary_df.copy()

    def to_en(sector):
        if pd.isna(sector):
            return ""
        return SECTOR_EN.get(sector, sector)

    df['Top_Q2_Sector_1_EN'] = df['Top_Q2_Sector_1'].apply(to_en)
    df['Top_Q2_Sector_2_EN'] = df['Top_Q2_Sector_2'].apply(to_en)

    return df

In [ ]:
# =========================================================
# Sharpening Index 補助集計：
# 主要部門限定（small-base problem 対策）
# =========================================================

def summarize_regional_sharpening_major(phi_all_df, min_share_t0=0.01):
    """
    初期シェアが一定以上の部門だけに限定した summary。
    min_share_t0=0.01 は 1985年 share 1%以上。
    """
    df = phi_all_df.copy()
    df = df[df['share_t0'] >= min_share_t0].copy()

    summary_rows = []

    for region in df['Region'].unique():
        d = df[df['Region'] == region].copy()
        q2 = d[d['quadrant'] == 'Q2 Structural Sharpening']
        active = q2[q2['phi'] > 1]
        defensive = q2[(q2['phi'] > 0) & (q2['phi'] <= 1)]

        q2_sorted = q2.sort_values('phi', ascending=False)

        summary_rows.append({
            'Region': region,
            'Region_Display': TRANSLATIONS['en']['regions'].get(region, region),
            'Num_Q2_Major': len(q2),
            'Num_Active_Major': len(active),
            'Num_Defensive_Major': len(defensive),
            'Mean_Phi_Q2_Major': q2['phi'].mean(),
            'Median_Phi_Q2_Major': q2['phi'].median(),
            'Top_Major_Q2_1': q2_sorted['sector'].iloc[0] if len(q2_sorted) >= 1 else None,
            'Top_Major_Q2_2': q2_sorted['sector'].iloc[1] if len(q2_sorted) >= 2 else None,
        })

    return pd.DataFrame(summary_rows)

# =========================================================
# Sharpening Index 補助集計：
# 重み付き Phi
# =========================================================

def add_weighted_phi(phi_all_df):
    df = phi_all_df.copy()

    # 1985 share で重み付け
    df['phi_weighted_t0'] = df['phi'] * df['share_t0']

    # 平均 share で重み付け
    df['avg_share'] = (df['share_t0'] + df['share_t1']) / 2
    df['phi_weighted_avgshare'] = df['phi'] * df['avg_share']

    return df

def summarize_regional_sharpening_weighted(phi_all_df):
    df = add_weighted_phi(phi_all_df)

    summary_rows = []

    for region in df['Region'].unique():
        d = df[df['Region'] == region].copy()
        q2 = d[d['quadrant'] == 'Q2 Structural Sharpening']
        active = q2[q2['phi'] > 1]
        defensive = q2[(q2['phi'] > 0) & (q2['phi'] <= 1)]

        q2_sorted = q2.sort_values('phi_weighted_avgshare', ascending=False)

        summary_rows.append({
            'Region': region,
            'Region_Display': TRANSLATIONS['en']['regions'].get(region, region),
            'Num_Q2_Sectors': len(q2),
            'Num_Active_Sharpening': len(active),
            'Num_Defensive_Sharpening': len(defensive),
            'Mean_Phi_Q2': q2['phi'].mean(),
            'Median_Phi_Q2': q2['phi'].median(),
            'Top_Weighted_Q2_1': q2_sorted['sector'].iloc[0] if len(q2_sorted) >= 1 else None,
            'Top_Weighted_Q2_2': q2_sorted['sector'].iloc[1] if len(q2_sorted) >= 2 else None,
        })

    return pd.DataFrame(summary_rows)

# =========================================================
# 実行：major summary / weighted summary
# =========================================================

major_summary_df = summarize_regional_sharpening_major(phi_all_df, min_share_t0=0.01)
weighted_summary_df = summarize_regional_sharpening_weighted(phi_all_df)

print("\n=== Major-sector summary (share_t0 >= 1%) ===")
display(major_summary_df)

print("\n=== Weighted summary ===")
display(weighted_summary_df)

# 保存
out_dir = os.path.join(SAVE_DIR, 'tables')
os.makedirs(out_dir, exist_ok=True)

major_summary_df.to_csv(
    os.path.join(out_dir, 'appendix_sharpening_index_major_summary_1985_2005.csv'),
    index=False, encoding='utf-8-sig'
)

weighted_summary_df.to_csv(
    os.path.join(out_dir, 'appendix_sharpening_index_weighted_summary_1985_2005.csv'),
    index=False, encoding='utf-8-sig'
)

In [ ]:
regions_for_comparison = ALL_REGIONS

phi_all_df_all = calculate_regional_sharpening_indices(
    regions=regions_for_comparison,
    year0='1985',
    year1='2005'
)

sharpening_summary_df_all = summarize_regional_sharpening(phi_all_df_all)
major_summary_df_all = summarize_regional_sharpening_major(phi_all_df_all, min_share_t0=0.01)
weighted_summary_df_all = summarize_regional_sharpening_weighted(phi_all_df_all)

display(sharpening_summary_df_all.sort_values('Region_Display'))
display(major_summary_df_all.sort_values('Region_Display'))
display(weighted_summary_df_all.sort_values('Region_Display'))

out_dir = os.path.join(SAVE_DIR, 'tables')
os.makedirs(out_dir, exist_ok=True)

phi_all_df_all.to_csv(
    os.path.join(out_dir, 'appendix_sharpening_index_all_regions_detail_1985_2005.csv'),
    index=False, encoding='utf-8-sig'
)

sharpening_summary_df_all.to_csv(
    os.path.join(out_dir, 'appendix_sharpening_index_all_regions_summary_1985_2005.csv'),
    index=False, encoding='utf-8-sig'
)

major_summary_df_all.to_csv(
    os.path.join(out_dir, 'appendix_sharpening_index_all_regions_major_summary_1985_2005.csv'),
    index=False, encoding='utf-8-sig'
)

weighted_summary_df_all.to_csv(
    os.path.join(out_dir, 'appendix_sharpening_index_all_regions_weighted_summary_1985_2005.csv'),
    index=False, encoding='utf-8-sig'
)

In [ ]:
def verify_factor_decomposition_additivity(region='shikoku', year1=1985, year2=2005):
    print("\n" + "="*70)
    print(f"【検証】要因分解の加法整合性: {region} {year1}->{year2}")
    print("="*70)

    analyzer = BridgeAnalyzer(region=region, year1=year1, year2=year2, lang='en', style='color')
    comp = analyzer.prepare_comparison_data()

    total_change_direct = comp['Contribution'].sum()
    share_sum = comp[analyzer.col_share].sum()
    rate_sum = comp[analyzer.col_rate].sum()

    # aggregate SSR の直接差
    sky1 = get_structure(year1, region)['skyline']
    sky2 = get_structure(year2, region)['skyline']
    agg_diff = calc_weighted_ssr(sky2) - calc_weighted_ssr(sky1)

    print(f"Direct aggregate change : {agg_diff:+.6f}")
    print(f"Share effect sum        : {share_sum:+.6f}")
    print(f"Rate effect sum         : {rate_sum:+.6f}")
    print(f"Share + Rate            : {share_sum + rate_sum:+.6f}")
    print(f"Residual                : {agg_diff - (share_sum + rate_sum):+.10f}")

verify_factor_decomposition_additivity('shikoku', 1985, 1990)
verify_factor_decomposition_additivity('shikoku', 1985, 2005)

In [ ]:
def build_sharpening_sensitivity_table(
    baseline_df,
    major_df,
    weighted_df
):
    def to_en(sector):
        if pd.isna(sector):
            return ""
        return SECTOR_EN.get(sector, sector)

    b = baseline_df.copy()
    m = major_df.copy()
    w = weighted_df.copy()

    out = (
        b[['Region', 'Region_Display', 'Num_Q2_Sectors', 'Mean_Phi_Q2']]
        .merge(
            m[['Region', 'Num_Q2_Major', 'Mean_Phi_Q2_Major']],
            on='Region', how='left'
        )
        .merge(
            w[['Region', 'Top_Weighted_Q2_1', 'Top_Weighted_Q2_2']],
            on='Region', how='left'
        )
    )

    out['Top_Weighted_Q2_1_EN'] = out['Top_Weighted_Q2_1'].apply(to_en)
    out['Top_Weighted_Q2_2_EN'] = out['Top_Weighted_Q2_2'].apply(to_en)

    out['Weighted_Representative_Sectors'] = out.apply(
        lambda r: ", ".join(
            [x for x in [r['Top_Weighted_Q2_1_EN'], r['Top_Weighted_Q2_2_EN']] if isinstance(x, str) and x.strip() != ""]
        ) if any(isinstance(x, str) and x.strip() != "" for x in [r['Top_Weighted_Q2_1_EN'], r['Top_Weighted_Q2_2_EN']]) else "---",
        axis=1
    )

    def interpret(row):
        bphi = row['Mean_Phi_Q2']
        mphi = row['Mean_Phi_Q2_Major']
        if pd.isna(bphi) and pd.isna(mphi):
            return "No major change"
        if pd.notna(bphi) and pd.notna(mphi):
            if abs(bphi - mphi) < 0.3:
                return "Broadly stable"
            elif mphi < bphi:
                return "Weaker after exclusion"
            else:
                return "Stronger after exclusion"
        return "Partially available"

    out['Interpretation'] = out.apply(interpret, axis=1)

    return out[
        [
            'Region',
            'Region_Display',
            'Num_Q2_Sectors',
            'Mean_Phi_Q2',
            'Num_Q2_Major',
            'Mean_Phi_Q2_Major',
            'Weighted_Representative_Sectors',
            'Interpretation'
        ]
    ]

sharpening_sensitivity_df = build_sharpening_sensitivity_table(
    baseline_df=sharpening_summary_df_all,
    major_df=major_summary_df_all,
    weighted_df=weighted_summary_df_all
)

display(sharpening_sensitivity_df)

sharpening_sensitivity_df.to_csv(
    os.path.join(SAVE_DIR, 'tables', 'appendix_sharpening_sensitivity_table.csv'),
    index=False, encoding='utf-8-sig'
)

## ダウンロード等

In [ ]:
# =========================================================
# 最終出力・エクスポート処理 セクション
# （CSVのExcel統合 ＆ 最終採用図のみZIP）
# =========================================================

def compile_csvs_to_excel(tables_dir):
    """指定ディレクトリ内の全CSVを読み込み、1つのExcelファイルに統合する"""
    print("\n>>> CSVファイルのExcel統合を開始します...")
    excel_output_path = os.path.join(tables_dir, 'all_results_combined.xlsx')
    csv_files = sorted(glob.glob(os.path.join(tables_dir, '*.csv')))

    if not csv_files:
        print("  ! tablesディレクトリにCSVファイルが見つかりません。")
        return

    with pd.ExcelWriter(excel_output_path, engine='openpyxl') as writer:
        for csv_file in csv_files:
            file_name = os.path.basename(csv_file)
            sheet_name = os.path.splitext(file_name)[0]

            if len(sheet_name) > 31:
                sheet_name = sheet_name[:31]

            try:
                df_temp = pd.read_csv(csv_file)
                df_temp.to_excel(writer, sheet_name=sheet_name, index=False)
                print(f"  ✓ Added sheet: {sheet_name}")
            except Exception as e:
                print(f"  x Failed to add {file_name}: {e}")

    print(f"  -> 統合Excelを保存しました: {excel_output_path}")


def collect_final_paper_figures(output_dir, zip_filename='paper_figures_final'):
    """
    最終採用図のみを SAVE_DIR 以下から収集して ZIP 化する
    """
    print("\n>>> Final paper figures を収集してZIPを作成します...")

    tmp_dir = os.path.join('/tmp', zip_filename)
    os.makedirs(tmp_dir, exist_ok=True)

    found, missing = [], []

    all_pngs = {}
    for root, _, fnames in os.walk(output_dir):
        for fname in fnames:
            if fname.lower().endswith('.png'):
                all_pngs[fname] = os.path.join(root, fname)

    for fname in FINAL_FIGURES:
        if fname in all_pngs:
            shutil.copy(all_pngs[fname], os.path.join(tmp_dir, fname))
            found.append(fname)
        else:
            missing.append(fname)

    print(f"  ✓ 収集成功: {len(found)} 件")
    if missing:
        print(f"  ❌ 見つからなかったファイル: {len(missing)} 件")
        for f in missing:
            print(f"     - {f}")

    zip_path_base = os.path.join(os.path.dirname(output_dir.rstrip('/')), zip_filename)
    shutil.make_archive(zip_path_base, 'zip', tmp_dir)
    print(f"  -> ZIPファイル作成完了: {zip_path_base}.zip")

    try:
        from google.colab import files
        print("  -> ダウンロードを開始します。")
        files.download(f'{zip_path_base}.zip')
    except ImportError:
        print("  ! ローカル環境のため自動ダウンロードはスキップしました。")

    shutil.rmtree(tmp_dir, ignore_errors=True)


# =========================================================
# 実行ブロック
# =========================================================
tables_target_dir = os.path.join(SAVE_DIR, 'tables')

compile_csvs_to_excel(tables_target_dir)
collect_final_paper_figures(SAVE_DIR)